# Assignment - Applications of Big Data (COMP3002)

**Student:** Phạm Minh Khôi  
**Student ID:** 22145599

__Due: Sunday, 09 August 2026 at 11:59 PM (VN Time)__


__Centre for Research in Mathematics And Data Science__

__School of Computer, Data and Mathematical Sciences__


## Description
For this assignment, you will need to create a complete program to perform sentiment classification for movie reviews from feature extraction to classification. For a given review, your program should be able to predict whether it is positive i.e. like the movie, or negative, i.e. dislike the movie. 

## About the data
You will use a large movie review dataset containing a set of 25,000 movie reviews for training, and 25,000 for testing. You can download the data from the vUWS under the assignments folder named aclImdb.zip. You can also visit the following website for more information about the dataset at http://ai.stanford.edu/~amaas/data/sentiment/ or download data directly from there. 

Unzip the data to your local directory. Enter the aclImdb/ directory created by the zip file (~500MB), you will find the following three items among others

- train/: feature files and raw text files for the training set
- test/: feature files and raw text files for the testing set
- README: the readme file for more information on the dataset

Read the README file carefully about the descriptions on the text files that contain the reviews and their naming convention. The directories we concern here are 

- ./aclImdb/train/pos: raw text files of positive reviews in the training set
- ./aclImdb/train/neg: raw text files of negative reviews in the training set
- ./aclImdb/test/pos: raw text files of positive reviews in the test set
- ./aclImdb/test/neg: raw text files of negative reviews in the test set

A full version of this data set is available at hdfs://hadoop.cdms.westernsydney.edu.au:9000/users/bigdata/Hadoop/imdb/fullversion. A tiny cut down version with much less number of files (20 each for training and 10 each for test) is also available at hdfs://hadoop.cdms.westernsydney.edu.au:9000/users/bigdata/Hadoop/imdb/tinyversion. The tiny version is for experimenting purpose. 

## Task 1. Feature extraction (15 points)
Use the map reduce model to convert all text data into matrices. Convert _ratings_ to vectors. These will be used for classification in Task 2. Use TF-IDF to vectorise the text files. See previous practical classes and lectures materials for TF-IDF. One step further though is to represent each text file (review) as a very long and sparse vector as the following. Assume `wordslist` is the final list of distinct words contained in all reviews and its length is $D$. Then each review will be a vector of length $D$, with each position associated with a word in `wordlist` and the value being either 0, if the corresponding word is absent in the review, or the word’s TF-IDF. For example, if `wordlist = [‘word1’, ‘word2’, ‘word3’, ‘word4’]` and review 1 contains `word1` and `word4`, then the vector representation of review 1 is [0.1, 0, 0, 0.4] assuming TF-IDF of `word1` and `word4` in review 1 is 0.1 and 0.4 respectively. Note that TF is calculated from one single document while IDF is obtained from all documents in the collection. 

### Requirements: 

1. Map reduce model is a must. Implement it using Hadoop streaming. All data are available on SCDMS HDFS. The recommendation is to work on the tiny version of the data to make the code work. You may try your code on the full version. However, the application to full version is not required. 
2. Generate two matrices: `training_data`, `test_data`, and two vectors, `training_targets`, `test_targets`. `training_data` should have $N$ rows and $D$ columns with each row corresponding to each review in the training set, where $N$ is the totally number of reviews in training set and $D$ is the total number of words. $N$ and $D$ vary depending on which version of the data you use. `training_targets` should have $N$ elements each of which is the rating of the review is for. `test_data` and `test_targets` are similar defined. 

Note:

<!--1.	If feature extraction is too difficult for you, you can use pre-computed bag of words features included in this data set. Refer to the Appendix and README file for details. However, if pre-computed features are used, __a 60% penalty__ will incur for this task, i.e. the maximum marks you can get from this task is 6 if you do so. -->
- Ratings scores extraction can be purely python. 
- Using map reduce model to extract TF-IDF is mandatory. If not used, a __50% penalty__ for this task will incur. There is no constraint on how to form the training and test matrices and vectors. There are many versions of TF-IDF. There is no preference for which version to use.
- You can use data frame (using `pandas` package) instead of matrices and vectors to store training and test data and targets. 

### Marking scheme for task 1:

<!--- Text file reading (1pt): read the text files for TF-IDF extraction. -->
- Rating scores extraction (3pts): parse the name of text files to extract ratings.
- TF-IDF extraction (10pts):  use map reduce model to extract TF-IDF for each text file.  
- Forming matrices and target vectors (or data frames) (2pts): collect TF-IDFs to form training and test data for task 2. 




## Task 1: submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

### 1.1 Environment and Dataset Setup

This section prepares the Python environment and checks whether the IMDb
dataset can be accessed correctly.

The dataset is divided into four main folders:

- `train/pos`: positive reviews used for training
- `train/neg`: negative reviews used for training
- `test/pos`: positive reviews used for testing
- `test/neg`: negative reviews used for testing

The notebook is stored in the same project folder as the `data` folder.
Therefore, a relative path is used instead of a computer-specific
absolute path. This makes the notebook easier to move and reproduce on
another computer.

In [1]:
# Import the os library.
# It helps Python work with folders and file paths.
import os


# The IMDb dataset is inside the project's data folder.
DATA_PATH = "data/aclImdb"


# Create the paths to the four required review folders.
train_positive_path = os.path.join(
    DATA_PATH,
    "train",
    "pos"
)

train_negative_path = os.path.join(
    DATA_PATH,
    "train",
    "neg"
)

test_positive_path = os.path.join(
    DATA_PATH,
    "test",
    "pos"
)

test_negative_path = os.path.join(
    DATA_PATH,
    "test",
    "neg"
)


# Display the folder in which this notebook is currently running.
print("Current working folder:")
print(os.getcwd())

print()
print("IMDb dataset folder:")
print(DATA_PATH)

Current working folder:
/media/sf_COMP3002

IMDb dataset folder:
data/aclImdb


#### 1.1.1 Dataset Folder Validation

Before processing the reviews, the program checks that all four required
folders exist.

It also counts the number of text files in each folder. This validation
prevents the later MapReduce and classification steps from running with
an incorrect or incomplete dataset path.

In [2]:
def count_text_files(folder_path):
    """
    Count the number of .txt files inside one folder.
    """

    # Start the count at zero.
    text_file_count = 0

    # Read every file name inside the folder.
    file_names = os.listdir(folder_path)

    # Check one file name at a time.
    for file_name in file_names:

        # Only count files that end with .txt.
        if file_name.endswith(".txt"):
            text_file_count = text_file_count + 1

    return text_file_count


# Store the four folders in a simple list.
review_folders = [
    ("Training positive", train_positive_path),
    ("Training negative", train_negative_path),
    ("Test positive", test_positive_path),
    ("Test negative", test_negative_path)
]


# Check each folder one at a time.
for folder_name, folder_path in review_folders:

    print(folder_name)
    print("Path:", folder_path)

    # Check whether the folder exists.
    if os.path.isdir(folder_path):

        number_of_reviews = count_text_files(
            folder_path
        )

        print("Status: Folder found")
        print("Text files:", number_of_reviews)

    else:

        print("Status: Folder not found")

    print("-" * 40)

Training positive
Path: data/aclImdb/train/pos
Status: Folder found
Text files: 12500
----------------------------------------
Training negative
Path: data/aclImdb/train/neg
Status: Folder found
Text files: 12500
----------------------------------------
Test positive
Path: data/aclImdb/test/pos
Status: Folder found
Text files: 12500
----------------------------------------
Test negative
Path: data/aclImdb/test/neg
Status: Folder found
Text files: 12500
----------------------------------------


#### Interpretation

All four required IMDb review folders were found successfully.

Each folder contains 12,500 text reviews, giving:

- 25,000 training reviews; and
- 25,000 test reviews.

This matches the dataset description provided in the assignment.
Therefore, the relative dataset path is correct and the review files are
ready for rating extraction and text processing.

### 1.2 Rating Extraction and Review Metadata

Each IMDb review file follows the naming format:

`reviewID_rating.txt`

For example, the file `123_9.txt` contains review ID `123` and rating
`9`.

This section extracts the numerical rating from every file name. It also
creates a metadata table so that each review can later be matched with
the correct TF-IDF feature vector.

The original numerical ratings are retained in Task 1. They will be
converted into positive and negative classes in Task 2.

#### 1.2.1 Extracting a Rating from One File Name

A small Python function is created to separate the rating from the review
file name.

The function performs three steps:

1. split the file name at the underscore;
2. keep the section containing the rating; and
3. remove the `.txt` extension.

The extracted rating is then converted from text into an integer.

In [3]:
def extract_rating(file_name):
    """
    Extract the numerical rating from an IMDb review file name.

    Example:
    123_9.txt -> 9
    """

    # Split the file name at the underscore.
    file_name_parts = file_name.split("_")

    # The second part contains the rating and file extension.
    rating_with_extension = file_name_parts[1]

    # Remove the .txt extension.
    rating_text = rating_with_extension.split(".")[0]

    # Convert the rating from text into an integer.
    rating_number = int(rating_text)

    return rating_number

In [4]:
# Get the names of files inside the training positive folder.
sample_file_names = os.listdir(
    train_positive_path
)

# Sort the file names to keep the result consistent.
sample_file_names = sorted(
    sample_file_names
)

# Select the first text file.
sample_file_name = None

for file_name in sample_file_names:

    if file_name.endswith(".txt"):
        sample_file_name = file_name
        break


# Extract and display its rating.
sample_rating = extract_rating(
    sample_file_name
)

print("Sample file name:", sample_file_name)
print("Extracted rating:", sample_rating)

Sample file name: 0_9.txt
Extracted rating: 9


#### Interpretation

The function successfully extracted the numerical rating from a real
IMDb review file name.

This confirms that the file naming convention can be used to obtain the
target value for every review without reading the review content.

#### 1.2.2 Building the Review Metadata Table

The program now reads all four review folders and records the following
information for every review:

- a unique document ID;
- training or test split;
- positive or negative source folder;
- original file name;
- numerical rating; and
- local file path.

The document ID includes the split, sentiment folder and file name. This
prevents files with similar names from being confused and will later
allow the Hadoop TF-IDF output to be matched with the correct rating.

In [5]:
# Import pandas for storing the review information in a table.
import pandas as pd


# Create an empty list.
# Each review will later be added to this list.
all_reviews = []


def add_reviews_from_folder(folder_path, split_name, sentiment_name):
    """
    Read all review file names from one folder
    and add their information to all_reviews.
    """

    # Read and sort all file names in the folder.
    file_names = os.listdir(
        folder_path
    )

    file_names = sorted(
        file_names
    )

    # Process one file name at a time.
    for file_name in file_names:

        # Ignore files that are not text reviews.
        if not file_name.endswith(".txt"):
            continue

        # Extract the numerical rating.
        rating = extract_rating(
            file_name
        )

        # Create a unique ID for the review.
        document_id = (
            split_name
            + "/"
            + sentiment_name
            + "/"
            + file_name
        )

        # Create the full local path to the review file.
        file_path = os.path.join(
            folder_path,
            file_name
        )

        # Store the information for one review.
        review_information = {
            "document_id": document_id,
            "split": split_name,
            "sentiment_folder": sentiment_name,
            "file_name": file_name,
            "rating": rating,
            "file_path": file_path
        }

        # Add this review to the main list.
        all_reviews.append(
            review_information
        )

In [6]:
# Clear old records before collecting the metadata again.
# This prevents duplicate rows if this cell is run more than once.
all_reviews.clear()


# Read positive training reviews.
add_reviews_from_folder(
    train_positive_path,
    "train",
    "pos"
)

# Read negative training reviews.
add_reviews_from_folder(
    train_negative_path,
    "train",
    "neg"
)

# Read positive test reviews.
add_reviews_from_folder(
    test_positive_path,
    "test",
    "pos"
)

# Read negative test reviews.
add_reviews_from_folder(
    test_negative_path,
    "test",
    "neg"
)


# Convert the list into a pandas DataFrame.
review_metadata = pd.DataFrame(
    all_reviews
)


print("Total reviews collected:", len(review_metadata))

display(
    review_metadata.head()
)

Total reviews collected: 50000


,document_id,split,sentiment_folder,file_name,rating,file_path
0,train/pos/0_9.txt,train,pos,0_9.txt,9,data/aclImdb/train/pos/0_9.txt
1,train/pos/10000_8.txt,train,pos,10000_8.txt,8,data/aclImdb/train/pos/10000_8.txt
2,train/pos/10001_10.txt,train,pos,10001_10.txt,10,data/aclImdb/train/pos/10001_10.txt
3,train/pos/10002_7.txt,train,pos,10002_7.txt,7,data/aclImdb/train/pos/10002_7.txt
4,train/pos/10003_8.txt,train,pos,10003_8.txt,8,data/aclImdb/train/pos/10003_8.txt


#### 1.2.3 Creating the Target Vectors

The complete metadata table is separated into training and test
metadata.

The `rating` column from each table is then converted into:

- `full_training_targets`; and
- `full_test_targets`.

These full-dataset vectors are used only for validation. The final Task 1 variables
`training_targets` and `test_targets` are created later for the 60-review tiny dataset.

In [7]:
# Select all training reviews.
training_metadata = review_metadata[
    review_metadata["split"] == "train"
].copy()

# Reset the row numbers from 0.
training_metadata = training_metadata.reset_index(
    drop=True
)


# Select all test reviews.
test_metadata = review_metadata[
    review_metadata["split"] == "test"
].copy()

# Reset the row numbers from 0.
test_metadata = test_metadata.reset_index(
    drop=True
)


# Extract the rating column as NumPy vectors.
full_training_targets = training_metadata[
    "rating"
].to_numpy()

full_test_targets = test_metadata[
    "rating"
].to_numpy()


# Display simple validation results.
print("Training reviews:", len(training_metadata))
print("Test reviews:", len(test_metadata))

print()
print("full_training_targets shape:", full_training_targets.shape)
print("full_test_targets shape:", full_test_targets.shape)

print()
print("First 10 training ratings:")
print(full_training_targets[:10])

print()
print("First 10 test ratings:")
print(full_test_targets[:10])

Training reviews: 25000
Test reviews: 25000

full_training_targets shape: (25000,)
full_test_targets shape: (25000,)

First 10 training ratings:
[ 9  8 10  7  8  8  7  7  7  7]

First 10 test ratings:
[10  7  9  8  8  9  8  7 10  8]


#### 1.2.4 Rating Validation

As a final check, the ratings extracted from the positive and negative
folders are examined separately.

In the labelled IMDb dataset:

- positive reviews use ratings from 7 to 10; and
- negative reviews use ratings from 1 to 4.

This validation helps confirm that the file names were parsed correctly
and that reviews were collected from the correct folders.

In [8]:
# Select ratings from positive review folders.
positive_ratings = review_metadata[
    review_metadata["sentiment_folder"] == "pos"
]["rating"]

# Select ratings from negative review folders.
negative_ratings = review_metadata[
    review_metadata["sentiment_folder"] == "neg"
]["rating"]


print("Positive rating range:")
print(
    positive_ratings.min(),
    "to",
    positive_ratings.max()
)

print()
print("Negative rating range:")
print(
    negative_ratings.min(),
    "to",
    negative_ratings.max()
)

Positive rating range:
7 to 10

Negative rating range:
1 to 4


#### Interpretation

The program collected 50,000 review records successfully:

- 25,000 training reviews; and
- 25,000 test reviews.

Both full-dataset target vectors contain 25,000 original numerical ratings. The
positive review ratings range from 7 to 10, while the negative review
ratings range from 1 to 4.

These results confirm that the ratings were extracted correctly and that
each review is linked to a unique document ID. The rating-extraction
component of Task 1 is therefore complete.

#### Hadoop Environment Requirement

This notebook was developed and tested in the course-provided local Hadoop
virtual machine.

Before running Sections 1.3 to 1.5, start HDFS and YARN from a terminal:

    start-dfs.sh
    start-yarn.sh

The notebook must also be opened from the shared project folder so that the
relative dataset path `data/aclImdb` is available.


### 1.3 Preparing IMDb Review Data in HDFS

The original IMDb dataset is currently stored in the local project
folder. However, Hadoop normally reads input data from the Hadoop
Distributed File System (HDFS), rather than directly from the local file
system.

To make development and testing faster, a small version of the dataset
is created first. This tiny dataset contains:

- 20 positive training reviews;
- 20 negative training reviews;
- 10 positive test reviews; and
- 10 negative test reviews.

The tiny dataset therefore contains 60 reviews in total. After it is
created locally, it is uploaded to HDFS for use in the MapReduce jobs.

#### 1.3.1 Creating a Tiny Local Dataset

The following code copies a fixed number of review files from each
original IMDb folder into a new local folder named `tiny_aclImdb`.

The file names are sorted before copying. This ensures that the same
reviews are selected every time the notebook is run, making the process
reproducible.

If an older tiny dataset already exists, it is removed and recreated.

In [9]:
# Import shutil.
# This library allows Python to copy and remove folders.
import shutil


# Local folder for the small test dataset.
TINY_DATA_PATH = "data/tiny_aclImdb"


# Remove the old tiny dataset if it already exists.
if os.path.isdir(TINY_DATA_PATH):
    shutil.rmtree(
        TINY_DATA_PATH
    )


# Create the four required tiny dataset folders.
tiny_train_positive_path = os.path.join(
    TINY_DATA_PATH,
    "train",
    "pos"
)

tiny_train_negative_path = os.path.join(
    TINY_DATA_PATH,
    "train",
    "neg"
)

tiny_test_positive_path = os.path.join(
    TINY_DATA_PATH,
    "test",
    "pos"
)

tiny_test_negative_path = os.path.join(
    TINY_DATA_PATH,
    "test",
    "neg"
)


os.makedirs(
    tiny_train_positive_path
)

os.makedirs(
    tiny_train_negative_path
)

os.makedirs(
    tiny_test_positive_path
)

os.makedirs(
    tiny_test_negative_path
)


print("Tiny dataset folders created successfully.")

Tiny dataset folders created successfully.


#### 1.3.2 Copying Reviews into the Tiny Dataset

A simple function is used to copy the first required number of text
reviews from an original folder into its matching tiny dataset folder.

Only files ending in `.txt` are copied. Other files, such as system or
metadata files, are ignored.

In [10]:
def copy_first_reviews(
    source_folder,
    destination_folder,
    number_of_reviews
):
    """
    Copy a selected number of text reviews
    from one folder into another folder.
    """

    # Read all file names from the source folder.
    file_names = os.listdir(
        source_folder
    )

    # Sort the names to make the selection reproducible.
    file_names = sorted(
        file_names
    )

    # Create an empty list for valid review files.
    text_file_names = []

    # Keep only files ending in .txt.
    for file_name in file_names:

        if file_name.endswith(".txt"):
            text_file_names.append(
                file_name
            )

    # Keep only the required number of reviews.
    selected_file_names = text_file_names[
        :number_of_reviews
    ]

    # Copy the selected reviews.
    for file_name in selected_file_names:

        source_file = os.path.join(
            source_folder,
            file_name
        )

        destination_file = os.path.join(
            destination_folder,
            file_name
        )

        shutil.copy(
            source_file,
            destination_file
        )

    return len(selected_file_names)

In [11]:
# Copy 20 positive training reviews.
train_positive_copied = copy_first_reviews(
    train_positive_path,
    tiny_train_positive_path,
    20
)

# Copy 20 negative training reviews.
train_negative_copied = copy_first_reviews(
    train_negative_path,
    tiny_train_negative_path,
    20
)

# Copy 10 positive test reviews.
test_positive_copied = copy_first_reviews(
    test_positive_path,
    tiny_test_positive_path,
    10
)

# Copy 10 negative test reviews.
test_negative_copied = copy_first_reviews(
    test_negative_path,
    tiny_test_negative_path,
    10
)


print(
    "Training positive reviews copied:",
    train_positive_copied
)

print(
    "Training negative reviews copied:",
    train_negative_copied
)

print(
    "Test positive reviews copied:",
    test_positive_copied
)

print(
    "Test negative reviews copied:",
    test_negative_copied
)

Training positive reviews copied: 20
Training negative reviews copied: 20
Test positive reviews copied: 10
Test negative reviews copied: 10


#### 1.3.3 Validating the Tiny Local Dataset

The newly created folders are checked before any files are uploaded to
HDFS.

This confirms that the correct number of reviews was copied into each
folder and prevents an incomplete dataset from being used by Hadoop.

In [12]:
# Count the review files in each tiny folder.
tiny_train_positive_count = count_text_files(
    tiny_train_positive_path
)

tiny_train_negative_count = count_text_files(
    tiny_train_negative_path
)

tiny_test_positive_count = count_text_files(
    tiny_test_positive_path
)

tiny_test_negative_count = count_text_files(
    tiny_test_negative_path
)


# Calculate the total number of tiny reviews.
total_tiny_reviews = (
    tiny_train_positive_count
    + tiny_train_negative_count
    + tiny_test_positive_count
    + tiny_test_negative_count
)


print(
    "Tiny train/pos:",
    tiny_train_positive_count
)

print(
    "Tiny train/neg:",
    tiny_train_negative_count
)

print(
    "Tiny test/pos:",
    tiny_test_positive_count
)

print(
    "Tiny test/neg:",
    tiny_test_negative_count
)

print()
print(
    "Total tiny reviews:",
    total_tiny_reviews
)

Tiny train/pos: 20
Tiny train/neg: 20
Tiny test/pos: 10
Tiny test/neg: 10

Total tiny reviews: 60


#### Interpretation

The tiny local dataset was created successfully.

It contains:

- 40 training reviews;
- 20 test reviews; and
- 60 reviews in total.

The folder structure matches the original IMDb dataset. The tiny dataset
is therefore ready to be uploaded to HDFS and processed using Hadoop
Streaming.

#### 1.3.4 Uploading the Tiny Dataset to HDFS

The tiny dataset currently exists in the local project folder. The next
step copies these files into HDFS.

The HDFS destination used in this notebook is:

`/users/hadoop/COMP3002/imdb_tiny`

An existing copy of this folder is removed before uploading the new
files. This ensures that rerunning the notebook does not create duplicate
or outdated data.

In [13]:
%%bash

# HDFS location for the tiny IMDb dataset.
HDFS_INPUT_PATH="/users/hadoop/COMP3002/imdb_tiny"


# Remove the old HDFS copy if it already exists.
hdfs dfs -rm -r -f "${HDFS_INPUT_PATH}"


# Create the four required HDFS folders.
hdfs dfs -mkdir -p "${HDFS_INPUT_PATH}/train/pos"
hdfs dfs -mkdir -p "${HDFS_INPUT_PATH}/train/neg"
hdfs dfs -mkdir -p "${HDFS_INPUT_PATH}/test/pos"
hdfs dfs -mkdir -p "${HDFS_INPUT_PATH}/test/neg"


# Upload the training reviews.
hdfs dfs -put \
    data/tiny_aclImdb/train/pos/*.txt \
    "${HDFS_INPUT_PATH}/train/pos/"

hdfs dfs -put \
    data/tiny_aclImdb/train/neg/*.txt \
    "${HDFS_INPUT_PATH}/train/neg/"


# Upload the test reviews.
hdfs dfs -put \
    data/tiny_aclImdb/test/pos/*.txt \
    "${HDFS_INPUT_PATH}/test/pos/"

hdfs dfs -put \
    data/tiny_aclImdb/test/neg/*.txt \
    "${HDFS_INPUT_PATH}/test/neg/"


echo "Tiny IMDb dataset uploaded to HDFS."

Deleted /users/hadoop/COMP3002/imdb_tiny
Tiny IMDb dataset uploaded to HDFS.


#### 1.3.5 Validating the Dataset in HDFS

The uploaded HDFS folders are now inspected.

The program counts all text review files stored under the HDFS input
path. The expected result is 60 files, matching the local tiny dataset.

In [14]:
%%bash

HDFS_INPUT_PATH="/users/hadoop/COMP3002/imdb_tiny"


echo "HDFS folder structure:"
hdfs dfs -ls -R "${HDFS_INPUT_PATH}" | awk 'NR <= 20 {print}'


echo
echo "Number of review files in HDFS:"

hdfs dfs -find "${HDFS_INPUT_PATH}" \
    -name "*.txt" \
    | wc -l

HDFS folder structure:
drwxr-xr-x   - hadoop supergroup          0 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test
drwxr-xr-x   - hadoop supergroup          0 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg
-rw-r--r--   1 hadoop supergroup        900 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/0_2.txt
-rw-r--r--   1 hadoop supergroup       1323 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/10000_4.txt
-rw-r--r--   1 hadoop supergroup       1273 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/10001_1.txt
-rw-r--r--   1 hadoop supergroup       1673 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/10002_3.txt
-rw-r--r--   1 hadoop supergroup       1473 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/10003_3.txt
-rw-r--r--   1 hadoop supergroup        662 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_tiny/test/neg/10004_2.txt
-rw-r--r--   1 hadoop supergroup        911 2026-07-17 20:37 /users/hadoop/COMP3002/imdb_

#### Interpretation

The tiny IMDb dataset was uploaded to HDFS successfully.

HDFS contains 60 review files, which matches the validated local tiny
dataset. The training/test and positive/negative directory structure was
also preserved.

The data is now available to Hadoop and can be used as the input for the
first MapReduce job.

### 1.4 MapReduce Job 1: Term Frequency for Each Review

The first MapReduce job calculates the term frequency of every word in
each review.

The job follows three stages:

1. the mapper identifies the source review and extracts its words;
2. Hadoop groups all words belonging to the same review; and
3. the reducer counts the words and calculates their term frequencies.

Term frequency is calculated as:

`term frequency = word count / total words in the review`

The output of this job contains:

- `document_id`
- `word`
- `word_count`
- `term_frequency`

This output will be used by the second MapReduce job to calculate inverse
document frequency and TF-IDF.

#### 1.4.1 Mapper: Extracting Words from Reviews

The mapper reads the review text from standard input.

For every review, it:

1. obtains the source file path from Hadoop;
2. creates a document ID using the final three path components;
3. converts the review text to lowercase;
4. removes HTML line-break markers;
5. extracts individual English words; and
6. emits the document ID and one word at a time.

For example, the review:

`Good movie, good acting!`

produces mapper records similar to:

    train/pos/example_9.txt    good
    train/pos/example_9.txt    movie
    train/pos/example_9.txt    good
    train/pos/example_9.txt    acting

Using the document ID as the mapper key allows Hadoop to send all words
from the same review to the same reducer.

In [15]:
%%writefile hadoopmapper1.py
#!/usr/bin/env python3

import os
import re
import sys


def get_document_id():
    """
    Get the ID of the review currently being processed.

    Example Hadoop path:
    /users/hadoop/COMP3002/imdb_tiny/train/pos/0_9.txt

    Returned ID:
    train/pos/0_9.txt
    """

    # Get the input file path from Hadoop.
    input_file_path = os.environ.get(
        "mapreduce_map_input_file"
    )

    # Some Hadoop versions use this older variable name.
    if input_file_path is None:
        input_file_path = os.environ.get(
            "map_input_file"
        )

    # Use a safe value if the path is unavailable.
    if input_file_path is None:
        return "unknown_file"

    # Use forward slashes consistently.
    input_file_path = input_file_path.replace(
        "\\",
        "/"
    )

    # Break the path into separate parts.
    path_parts = input_file_path.split("/")

    # Keep the split, sentiment folder and file name.
    document_id = "/".join(
        path_parts[-3:]
    )

    return document_id


def extract_words(review_text):
    """
    Convert review text into lowercase English words.
    """

    # Convert all letters to lowercase.
    review_text = review_text.lower()

    # Remove the common HTML line-break marker.
    review_text = review_text.replace(
        "<br />",
        " "
    )

    # Extract sequences containing letters from a to z.
    words = re.findall(
        r"[a-z]+",
        review_text
    )

    return words


# Identify the review being processed.
document_id = get_document_id()


# Hadoop sends the review text one line at a time.
for line in sys.stdin:

    words = extract_words(
        line
    )

    # Emit one record for every word occurrence.
    for word in words:

        print(
            document_id
            + "\t"
            + word
        )

Overwriting hadoopmapper1.py


#### 1.4.2 Reducer: Counting Words and Calculating TF

Hadoop sorts the mapper output by document ID before sending it to the
reducer. Therefore, all words belonging to the same review arrive
together.

For each review, the reducer:

1. counts the total number of words;
2. counts how many times each distinct word appears; and
3. divides each word count by the total number of words.

For example:

    good
    movie
    good
    acting

contains four words in total. The reducer produces:

    good      2    0.500000
    movie     1    0.250000
    acting    1    0.250000

In [16]:
%%writefile hadoopreducer1.py
#!/usr/bin/env python3

import sys


def print_document_results(
    document_id,
    word_counts,
    total_words
):
    """
    Print the word count and term frequency
    for every word in one review.
    """

    # Sort the words to keep the output consistent.
    words = sorted(
        word_counts.keys()
    )

    for word in words:

        word_count = word_counts[word]

        term_frequency = (
            word_count
            / total_words
        )

        print(
            document_id
            + "\t"
            + word
            + "\t"
            + str(word_count)
            + "\t"
            + format(term_frequency, ".6f")
        )


# Store the review currently being processed.
current_document = None

# Store word counts for the current review.
current_word_counts = {}

# Store the total number of words in the current review.
current_total_words = 0


# Read the sorted mapper output.
for line in sys.stdin:

    line = line.strip()

    # Ignore empty lines.
    if line == "":
        continue

    # Separate the document ID and word.
    document_id, word = line.split(
        "\t",
        1
    )

    # Continue processing the same review.
    if document_id == current_document:

        current_total_words = (
            current_total_words + 1
        )

        if word in current_word_counts:

            current_word_counts[word] = (
                current_word_counts[word] + 1
            )

        else:

            current_word_counts[word] = 1

    else:

        # Print the completed previous review.
        if current_document is not None:

            print_document_results(
                current_document,
                current_word_counts,
                current_total_words
            )

        # Begin processing a new review.
        current_document = document_id
        current_word_counts = {
            word: 1
        }
        current_total_words = 1


# Print the final review after the loop finishes.
if current_document is not None:

    print_document_results(
        current_document,
        current_word_counts,
        current_total_words
    )

Overwriting hadoopreducer1.py


#### 1.4.3 Local Validation of the Mapper and Reducer

Before running the scripts through Hadoop, they are tested using one
small artificial review.

This test follows the same data flow as Hadoop:

1. the sample review is sent to the mapper;
2. the mapper output is sorted; and
3. the sorted records are sent to the reducer.

The sample contains four words, allowing the expected term frequencies
to be checked manually.

In [17]:
%%bash

# Give the mapper a sample Hadoop-style input path.
export mapreduce_map_input_file="/users/hadoop/COMP3002/imdb_tiny/train/pos/example_9.txt"


# Send a small review through the mapper and reducer.
printf "Good movie, good acting!\n" \
    | python3 hadoopmapper1.py \
    | sort \
    | python3 hadoopreducer1.py

train/pos/example_9.txt	acting	1	0.250000
train/pos/example_9.txt	good	2	0.500000
train/pos/example_9.txt	movie	1	0.250000


#### Interpretation

The local test produced the expected result.

The sample review contains four words in total. The word `good` appears
twice and therefore has a term frequency of 0.50. The words `movie` and
`acting` each appear once and have term frequencies of 0.25.

This confirms that the mapper correctly extracts words and that the
reducer correctly calculates word counts and term frequencies before the
scripts are executed on HDFS data.

#### 1.4.4 Hadoop Streaming Execution

The validated mapper and reducer are now executed through Hadoop
Streaming.

The job reads the 60 review files stored under:

`/users/hadoop/COMP3002/imdb_tiny`

Recursive input processing is enabled because the files are stored
inside nested training, test, positive and negative folders.

The output is stored at:

`/users/hadoop/COMP3002/job1_term_frequency`

The previous output folder is removed before execution because Hadoop
does not overwrite an existing output directory.

In [18]:
%%bash

# HDFS input and output locations.
INPUT_PATH="/users/hadoop/COMP3002/imdb_tiny"

OUTPUT_PATH="/users/hadoop/COMP3002/job1_term_frequency"


# Find the Hadoop Streaming library installed in this VM.
STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


echo "Hadoop Streaming library:"
echo "${STREAMING_JAR}"


# Stop if the Streaming library cannot be found.
if [ -z "${STREAMING_JAR}" ]; then

    echo "ERROR: Hadoop Streaming library was not found."
    exit 1

fi


# Remove the previous output folder.
hdfs dfs -rm -r -f "${OUTPUT_PATH}"


# Run MapReduce Job 1.
hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.input.fileinputformat.input.dir.recursive=true \
    -D mapreduce.job.reduces=1 \
    -files hadoopmapper1.py,hadoopreducer1.py \
    -mapper "python3 hadoopmapper1.py" \
    -reducer "python3 hadoopreducer1.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}"

Hadoop Streaming library:
/home/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.1.jar
Deleted /users/hadoop/COMP3002/job1_term_frequency
packageJobJar: [/tmp/hadoop-unjar2099762501751016477/] [] /tmp/streamjob4506636993495881167.jar tmpDir=null


2026-07-17 20:37:34,976 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:37:35,493 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:37:35,905 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0011
2026-07-17 20:37:36,567 INFO mapred.FileInputFormat: Total input files to process : 60
2026-07-17 20:37:36,887 INFO mapreduce.JobSubmitter: number of splits:60
2026-07-17 20:37:37,275 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0011
2026-07-17 20:37:37,275 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:37:37,740 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:37:37,742 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:37:37,820 INFO impl.YarnClientImpl: Submitted application application_178426244

#### 1.4.5 Validating the MapReduce Output

The output of MapReduce Job 1 is inspected to confirm that:

- records were produced successfully;
- all 60 reviews appear in the output; and
- each record contains a document ID, word, word count and term
  frequency.

The number of output records is larger than 60 because every review
contains many distinct words.

In [19]:
%%bash

OUTPUT_PATH="/users/hadoop/COMP3002/job1_term_frequency"


echo "First 15 output records:"
echo

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | awk 'NR <= 15 {print}'


echo
echo "Total TF records:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | wc -l


echo
echo "Unique reviews in the output:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | cut -f1 \
    | sort -u \
    | wc -l

First 15 output records:

test/neg/0_2.txt	a	3	0.017544
test/neg/0_2.txt	about	3	0.017544
test/neg/0_2.txt	again	1	0.005848
test/neg/0_2.txt	all	2	0.011696
test/neg/0_2.txt	an	1	0.005848
test/neg/0_2.txt	and	3	0.017544
test/neg/0_2.txt	any	1	0.005848
test/neg/0_2.txt	anyone	1	0.005848
test/neg/0_2.txt	appears	1	0.005848
test/neg/0_2.txt	are	4	0.023392
test/neg/0_2.txt	around	1	0.005848
test/neg/0_2.txt	as	1	0.005848
test/neg/0_2.txt	ashton	1	0.005848
test/neg/0_2.txt	aside	1	0.005848
test/neg/0_2.txt	be	2	0.011696

Total TF records:
8251

Unique reviews in the output:
60


#### Interpretation

MapReduce Job 1 completed successfully and produced term-frequency
records for all 60 reviews in the tiny dataset.

Each output row contains:

- the unique review ID;
- one word from that review;
- the number of times the word appeared; and
- its term frequency within the review.

The validation found 60 unique document IDs, confirming that no review
was omitted during Hadoop processing.

The output is now ready for MapReduce Job 2, which will calculate
document frequency, inverse document frequency and final TF-IDF values.

### 1.5 MapReduce Job 2: IDF and TF-IDF Calculation

The second MapReduce job calculates document frequency, inverse document
frequency and TF-IDF for every word.

Document frequency (`DF`) is the number of reviews containing a word.
Because MapReduce Job 1 already produces only one record for each
document-word pair, the reducer can calculate DF by counting the number
of records belonging to each word.

This implementation uses smoothed inverse document frequency:

$$
IDF(w)
=
\log\left(
\frac{N+1}{DF(w)+1}
\right)+1
$$

where:

- $N$ is the total number of reviews in the collection;
- $DF(w)$ is the number of reviews containing word $w$; and
- adding 1 prevents division by zero and avoids an IDF value of zero.

TF-IDF is then calculated as:

$$
TFIDF(d,w)=TF(d,w)\times IDF(w)
$$

For the tiny dataset, $N=60$.

#### 1.5.1 Mapper: Grouping Records by Word

The mapper reads the output produced by MapReduce Job 1.

Each input record contains:

- document ID;
- word;
- word count; and
- term frequency.

The mapper changes the key from the document ID to the word. This allows
Hadoop to group together all reviews containing the same word.

For example, the Job 1 records:

    train/pos/review1_9.txt    good    2    0.500000
    train/neg/review2_2.txt    good    1    0.250000

are emitted with `good` as their shared key.

In [20]:
%%writefile hadoopmapper2.py
#!/usr/bin/env python3

import sys


# Read MapReduce Job 1 output one line at a time.
for line in sys.stdin:

    # Remove the line-break character.
    line = line.strip()

    # Ignore empty lines.
    if line == "":
        continue

    # Separate the four Job 1 fields.
    fields = line.split(
        "\t"
    )

    # Ignore an invalid record.
    if len(fields) != 4:
        continue

    document_id = fields[0]
    word = fields[1]
    word_count = fields[2]
    term_frequency = fields[3]

    # Use the word as the Hadoop key.
    print(
        word
        + "\t"
        + document_id
        + "\t"
        + word_count
        + "\t"
        + term_frequency
    )

Overwriting hadoopmapper2.py


#### 1.5.2 Reducer: Calculating DF, IDF and TF-IDF

Hadoop sorts the mapper output by word before sending it to the reducer.
Therefore, all records belonging to the same word arrive together.

For each word, the reducer:

1. counts how many documents contain the word;
2. calculates its smoothed IDF;
3. multiplies each term frequency by the IDF; and
4. produces one TF-IDF record for every document-word pair.

The final output contains seven fields:

- `document_id`
- `word`
- `word_count`
- `term_frequency`
- `document_frequency`
- `inverse_document_frequency`
- `tfidf`

In [21]:
%%writefile hadoopreducer2.py
#!/usr/bin/env python3

import math
import os
import sys


# Read the total number of documents from Hadoop.
total_documents_text = os.environ.get(
    "TOTAL_DOCUMENTS"
)

# Use 60 as a safe default for the tiny dataset.
if total_documents_text is None:
    total_documents = 60
else:
    total_documents = int(
        total_documents_text
    )


def print_tfidf_results(
    word,
    document_records
):
    """
    Calculate and print TF-IDF results for one word.
    """

    # One record represents one document containing the word.
    document_frequency = len(
        document_records
    )

    # Calculate smoothed IDF.
    inverse_document_frequency = math.log(
        (total_documents + 1)
        / (document_frequency + 1)
    ) + 1

    # Sort records by document ID for consistent output.
    document_records = sorted(
        document_records
    )

    # Calculate TF-IDF for each document.
    for record in document_records:

        document_id = record[0]
        word_count = record[1]
        term_frequency = record[2]

        tfidf = (
            term_frequency
            * inverse_document_frequency
        )

        print(
            document_id
            + "\t"
            + word
            + "\t"
            + str(word_count)
            + "\t"
            + format(term_frequency, ".6f")
            + "\t"
            + str(document_frequency)
            + "\t"
            + format(
                inverse_document_frequency,
                ".6f"
            )
            + "\t"
            + format(tfidf, ".6f")
        )


# Store the word currently being processed.
current_word = None

# Store all document records for the current word.
current_document_records = []


# Read sorted mapper output.
for line in sys.stdin:

    line = line.strip()

    # Ignore empty lines.
    if line == "":
        continue

    # Separate the mapper output fields.
    fields = line.split(
        "\t"
    )

    # Ignore an invalid record.
    if len(fields) != 4:
        continue

    word = fields[0]
    document_id = fields[1]
    word_count = int(
        fields[2]
    )
    term_frequency = float(
        fields[3]
    )

    # Store the current document information.
    document_record = (
        document_id,
        word_count,
        term_frequency
    )

    # Continue collecting records for the same word.
    if word == current_word:

        current_document_records.append(
            document_record
        )

    else:

        # Print the completed previous word.
        if current_word is not None:

            print_tfidf_results(
                current_word,
                current_document_records
            )

        # Begin collecting a new word.
        current_word = word
        current_document_records = [
            document_record
        ]


# Print the final word after the loop ends.
if current_word is not None:

    print_tfidf_results(
        current_word,
        current_document_records
    )

Overwriting hadoopreducer2.py


#### 1.5.3 Local Validation of MapReduce Job 2

Before running the second job through Hadoop, the mapper and reducer are
tested using four artificial Job 1 records.

The sample contains three documents. The word `good` appears in all
three documents, while `movie` appears in only one document.

Therefore:

- `good` should have a lower IDF because it is common; and
- `movie` should have a higher IDF because it is less common.

This test confirms that document frequency and TF-IDF are calculated as
expected.

In [22]:
%%bash

# Create a small artificial Job 1 output file.
cat > sample_job1_output.txt << 'EOF'
train/pos/document1_9.txt	good	2	0.500000
train/neg/document2_2.txt	good	1	0.250000
test/pos/document3_8.txt	good	1	0.200000
train/pos/document1_9.txt	movie	1	0.250000
EOF


# The artificial sample contains three documents.
export TOTAL_DOCUMENTS=3


# Run the sample through Mapper 2 and Reducer 2.
cat sample_job1_output.txt \
    | python3 hadoopmapper2.py \
    | sort \
    | python3 hadoopreducer2.py


# Remove the temporary test file.
rm sample_job1_output.txt

test/pos/document3_8.txt	good	1	0.200000	3	1.000000	0.200000
train/neg/document2_2.txt	good	1	0.250000	3	1.000000	0.250000
train/pos/document1_9.txt	good	2	0.500000	3	1.000000	0.500000
train/pos/document1_9.txt	movie	1	0.250000	1	1.693147	0.423287


#### Interpretation

The local test produced the expected TF-IDF behaviour.

The word `good` appeared in all three sample documents. Its document
frequency was therefore 3 and its IDF was 1.00.

The word `movie` appeared in only one document. Its lower document
frequency produced a higher IDF of approximately 1.69.

This demonstrates the purpose of IDF: words appearing in fewer documents
receive greater importance than words appearing throughout the
collection.

The mapper and reducer are therefore ready to run through Hadoop
Streaming.

#### 1.5.4 Hadoop Streaming Execution

MapReduce Job 2 now reads the term-frequency output from Job 1.

The input directory is:

`/users/hadoop/COMP3002/job1_term_frequency`

The final TF-IDF output is stored at:

`/users/hadoop/COMP3002/job2_tfidf`

The total number of documents is passed to the reducer through the
environment variable `TOTAL_DOCUMENTS`. Its value is 60 for the tiny
dataset.

In [23]:
%%bash

# Input and output HDFS locations.
INPUT_PATH="/users/hadoop/COMP3002/job1_term_frequency"

OUTPUT_PATH="/users/hadoop/COMP3002/job2_tfidf"


# Find the Hadoop Streaming library.
STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


echo "Hadoop Streaming library:"
echo "${STREAMING_JAR}"


# Stop if the Streaming library cannot be found.
if [ -z "${STREAMING_JAR}" ]; then

    echo "ERROR: Hadoop Streaming library was not found."
    exit 1

fi


# Remove the previous Job 2 output.
hdfs dfs -rm -r -f "${OUTPUT_PATH}"


# Run MapReduce Job 2.
hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.job.reduces=1 \
    -files hadoopmapper2.py,hadoopreducer2.py \
    -cmdenv TOTAL_DOCUMENTS=60 \
    -mapper "python3 hadoopmapper2.py" \
    -reducer "python3 hadoopreducer2.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}"

Hadoop Streaming library:
/home/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.1.jar
Deleted /users/hadoop/COMP3002/job2_tfidf
packageJobJar: [/tmp/hadoop-unjar4680999181081368280/] [] /tmp/streamjob6456437867136689803.jar tmpDir=null


2026-07-17 20:41:16,763 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:41:17,367 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:41:17,667 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0012
2026-07-17 20:41:18,435 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-17 20:41:18,634 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-17 20:41:18,870 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0012
2026-07-17 20:41:18,870 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:41:19,360 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:41:19,362 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:41:19,486 INFO impl.YarnClientImpl: Submitted application application_17842624431

#### 1.5.5 Validating the Final TF-IDF Output

The final Hadoop output is inspected before it is converted into
matrices.

The validation checks:

- a sample of the output records;
- the total number of TF-IDF records;
- the number of unique reviews;
- the number of distinct words; and
- whether every output row contains the expected seven fields.

All 60 reviews should appear in the output.

In [24]:
%%bash

OUTPUT_PATH="/users/hadoop/COMP3002/job2_tfidf"


echo "First 15 TF-IDF records:"
echo

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | awk 'NR <= 15 {print}'


echo
echo "Total TF-IDF records:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | wc -l


echo
echo "Unique reviews:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | cut -f1 \
    | sort -u \
    | wc -l


echo
echo "Distinct words:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | cut -f2 \
    | sort -u \
    | wc -l


echo
echo "Records with an incorrect number of fields:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" \
    | awk -F '\t' '
        NF != 7 {
            invalid_records = invalid_records + 1
        }

        END {
            print invalid_records + 0
        }
    '

First 15 TF-IDF records:

test/neg/0_2.txt	a	3	0.017544	57	1.050431	0.018429
test/neg/10000_4.txt	a	4	0.016598	57	1.050431	0.017435
test/neg/10001_1.txt	a	6	0.026786	57	1.050431	0.028137
test/neg/10002_3.txt	a	8	0.025078	57	1.050431	0.026343
test/neg/10003_3.txt	a	4	0.016878	57	1.050431	0.017729
test/neg/10004_2.txt	a	3	0.025424	57	1.050431	0.026706
test/neg/10005_2.txt	a	5	0.030675	57	1.050431	0.032222
test/neg/10006_2.txt	a	5	0.034722	57	1.050431	0.036473
test/neg/10007_4.txt	a	7	0.054264	57	1.050431	0.057001
test/neg/10008_4.txt	a	6	0.035928	57	1.050431	0.037740
test/pos/0_10.txt	a	2	0.013072	57	1.050431	0.013731
test/pos/10000_7.txt	a	13	0.036932	57	1.050431	0.038795
test/pos/10001_9.txt	a	3	0.012146	57	1.050431	0.012759
test/pos/10002_8.txt	a	6	0.046875	57	1.050431	0.049239
test/pos/10003_8.txt	a	7	0.034146	57	1.050431	0.035868

Total TF-IDF records:
8251

Unique reviews:
60

Distinct words:
3127

Records with an incorrect number of fields:
0


#### Interpretation

MapReduce Job 2 completed successfully.

The output contains TF-IDF records for all 60 reviews in the tiny IMDb
dataset. Every output record contains the expected seven fields, and no
malformed records were detected.

Common words have higher document frequencies and lower IDF values,
while less common words receive higher IDF values. Each term frequency
was multiplied by its corresponding IDF to produce the final TF-IDF
feature value.

The mandatory MapReduce TF-IDF extraction stage is now complete. The
next section will collect these sparse TF-IDF records and form:

- `training_data`
- `test_data`
- `training_targets`
- `test_targets`

### 1.6 Forming the Training and Test Data

MapReduce Job 2 produced one sparse TF-IDF record for every
document-word pair where the word appears in the review.

Machine-learning models require these records to be reorganised into a
matrix structure:

- each row represents one review;
- each column represents one distinct word; and
- each value is the word's TF-IDF value in that review.

If a word does not appear in a review, its matrix value is zero.

Because the tiny dataset is used, the final objects contain:

- 40 training reviews; and
- 20 test reviews.

The same word list and column order are used for both matrices.

#### 1.6.1 Copying the TF-IDF Output to the Local Project

The final TF-IDF records are currently stored in HDFS.

The following command combines the Hadoop output files into one local
tab-separated file named:

`outputs/job2_tfidf.tsv`

Storing a local copy allows pandas to read and reorganise the records
into training and test matrices.

In [25]:
%%bash

# HDFS folder produced by MapReduce Job 2.
HDFS_TFIDF_PATH="/users/hadoop/COMP3002/job2_tfidf"

# Local folder and file used by this notebook.
LOCAL_OUTPUT_FOLDER="outputs"
LOCAL_TFIDF_FILE="${LOCAL_OUTPUT_FOLDER}/job2_tfidf.tsv"


# Create the local output folder if it does not exist.
mkdir -p "${LOCAL_OUTPUT_FOLDER}"


# Remove an older local copy if it exists.
rm -f "${LOCAL_TFIDF_FILE}"


# Combine the Hadoop output into one local file.
hdfs dfs -getmerge \
    "${HDFS_TFIDF_PATH}" \
    "${LOCAL_TFIDF_FILE}"


echo "TF-IDF output copied successfully."
echo
echo "Local file:"
echo "${LOCAL_TFIDF_FILE}"

echo
echo "Number of TF-IDF records:"
wc -l < "${LOCAL_TFIDF_FILE}"

TF-IDF output copied successfully.

Local file:
outputs/job2_tfidf.tsv

Number of TF-IDF records:
8251


#### 1.6.2 Loading the TF-IDF Records

The local Hadoop output is loaded into a pandas DataFrame.

Each record contains seven fields:

- `document_id`
- `word`
- `word_count`
- `term_frequency`
- `document_frequency`
- `inverse_document_frequency`
- `tfidf`

Only the document ID, word and TF-IDF value are required when forming
the feature matrices. The remaining fields are retained for validation
and interpretation.

In [26]:
# Define the names of the seven Hadoop output fields.
tfidf_column_names = [
    "document_id",
    "word",
    "word_count",
    "term_frequency",
    "document_frequency",
    "inverse_document_frequency",
    "tfidf"
]


# Read the tab-separated Hadoop output.
tfidf_records = pd.read_csv(
    "outputs/job2_tfidf.tsv",
    sep="\t",
    names=tfidf_column_names
)


print(
    "TF-IDF records loaded:",
    len(tfidf_records)
)

print()
print("Columns:")
print(tfidf_records.columns.tolist())

display(
    tfidf_records.head()
)

TF-IDF records loaded: 8251

Columns:
['document_id', 'word', 'word_count', 'term_frequency', 'document_frequency', 'inverse_document_frequency', 'tfidf']


,document_id,word,word_count,term_frequency,document_frequency,inverse_document_frequency,tfidf
0,test/neg/0_2.txt,a,3,0.017544,57,1.050431,0.018429
1,test/neg/10000_4.txt,a,4,0.016598,57,1.050431,0.017435
2,test/neg/10001_1.txt,a,6,0.026786,57,1.050431,0.028137
3,test/neg/10002_3.txt,a,8,0.025078,57,1.050431,0.026343
4,test/neg/10003_3.txt,a,4,0.016878,57,1.050431,0.017729


#### 1.6.3 Validating the Loaded TF-IDF Records

Before forming the matrices, the TF-IDF table is checked for:

- missing values;
- duplicate document-word pairs;
- the number of unique reviews; and
- the number of distinct words.

Every document-word pair should appear only once because the Hadoop
reducers have already aggregated their values.

In [27]:
# Count missing values in the complete TF-IDF table.
missing_values = tfidf_records.isna().sum().sum()


# Check whether a document-word pair appears more than once.
duplicate_pairs = tfidf_records.duplicated(
    subset=[
        "document_id",
        "word"
    ]
).sum()


# Count unique reviews.
number_of_documents = tfidf_records[
    "document_id"
].nunique()


# Count distinct words.
number_of_words = tfidf_records[
    "word"
].nunique()


print("Missing values:", missing_values)
print("Duplicate document-word pairs:", duplicate_pairs)
print("Unique reviews:", number_of_documents)
print("Distinct words:", number_of_words)

Missing values: 0
Duplicate document-word pairs: 0
Unique reviews: 60
Distinct words: 3127


#### Interpretation

The Hadoop TF-IDF output was loaded successfully.

The table contains records for all 60 reviews. No missing values or
duplicate document-word pairs were detected. This confirms that each
TF-IDF value uniquely represents one word in one review.

The number of distinct words reported above becomes the number of
columns in the training and test feature matrices.

#### 1.6.4 Creating Metadata for the Tiny Dataset

The earlier metadata table described the complete 50,000-review
dataset. However, the Hadoop jobs were executed on the 60-review tiny
dataset.

A new metadata table is therefore created specifically for the tiny
dataset. This ensures that:

- each TF-IDF matrix row matches the correct review;
- each review matches the correct rating; and
- training and test records remain clearly separated.

In [28]:
def collect_reviews_from_folder(
    folder_path,
    split_name,
    sentiment_name
):
    """
    Collect metadata from one tiny review folder.
    """

    # Create an empty list for this folder.
    folder_reviews = []

    # Read the file names in a consistent order.
    file_names = os.listdir(
        folder_path
    )

    file_names = sorted(
        file_names
    )

    # Process one review file at a time.
    for file_name in file_names:

        # Ignore files that are not text reviews.
        if not file_name.endswith(".txt"):
            continue

        # Create the same document ID used by Hadoop.
        document_id = (
            split_name
            + "/"
            + sentiment_name
            + "/"
            + file_name
        )

        # Extract the rating from the file name.
        rating = extract_rating(
            file_name
        )

        # Store the review information.
        review_information = {
            "document_id": document_id,
            "split": split_name,
            "sentiment_folder": sentiment_name,
            "file_name": file_name,
            "rating": rating
        }

        folder_reviews.append(
            review_information
        )

    return folder_reviews

In [29]:
# Create an empty list for all tiny review records.
tiny_review_records = []


# Collect the 20 positive training reviews.
tiny_review_records.extend(
    collect_reviews_from_folder(
        tiny_train_positive_path,
        "train",
        "pos"
    )
)


# Collect the 20 negative training reviews.
tiny_review_records.extend(
    collect_reviews_from_folder(
        tiny_train_negative_path,
        "train",
        "neg"
    )
)


# Collect the 10 positive test reviews.
tiny_review_records.extend(
    collect_reviews_from_folder(
        tiny_test_positive_path,
        "test",
        "pos"
    )
)


# Collect the 10 negative test reviews.
tiny_review_records.extend(
    collect_reviews_from_folder(
        tiny_test_negative_path,
        "test",
        "neg"
    )
)


# Convert the records into a DataFrame.
tiny_review_metadata = pd.DataFrame(
    tiny_review_records
)


print(
    "Tiny review records:",
    len(tiny_review_metadata)
)

display(
    tiny_review_metadata.head()
)

Tiny review records: 60


,document_id,split,sentiment_folder,file_name,rating
0,train/pos/0_9.txt,train,pos,0_9.txt,9
1,train/pos/10000_8.txt,train,pos,10000_8.txt,8
2,train/pos/10001_10.txt,train,pos,10001_10.txt,10
3,train/pos/10002_7.txt,train,pos,10002_7.txt,7
4,train/pos/10003_8.txt,train,pos,10003_8.txt,8


#### 1.6.5 Matching Metadata with Hadoop Output

The document IDs collected from the local tiny dataset are compared
with the document IDs produced by Hadoop.

The two sets must match exactly. A mismatch would indicate that a review
was omitted, renamed or processed incorrectly.

In [30]:
# Document IDs expected from the local tiny dataset.
expected_document_ids = set(
    tiny_review_metadata[
        "document_id"
    ]
)


# Document IDs found in the Hadoop TF-IDF output.
hadoop_document_ids = set(
    tfidf_records[
        "document_id"
    ]
)


# Find expected documents missing from Hadoop output.
missing_documents = (
    expected_document_ids
    - hadoop_document_ids
)


# Find unexpected documents in Hadoop output.
unexpected_documents = (
    hadoop_document_ids
    - expected_document_ids
)


print(
    "Expected documents:",
    len(expected_document_ids)
)

print(
    "Hadoop documents:",
    len(hadoop_document_ids)
)

print(
    "Missing documents:",
    len(missing_documents)
)

print(
    "Unexpected documents:",
    len(unexpected_documents)
)

Expected documents: 60
Hadoop documents: 60
Missing documents: 0
Unexpected documents: 0


#### 1.6.6 Creating the Word List and TF-IDF Table

A sorted list of all distinct words is created. This list defines the
column order used by both feature matrices.

The sparse Hadoop records are then reshaped into a complete table:

- review IDs become rows;
- words become columns; and
- TF-IDF scores become cell values.

Missing document-word combinations are filled with zero because the word
does not appear in that review.

In [31]:
# Create a sorted list of all distinct words.
wordlist = sorted(
    tfidf_records[
        "word"
    ].unique().tolist()
)


# Reshape sparse TF-IDF records into a review-by-word table.
complete_tfidf_table = tfidf_records.pivot(
    index="document_id",
    columns="word",
    values="tfidf"
)


# Use the sorted word list as the exact column order.
complete_tfidf_table = complete_tfidf_table.reindex(
    columns=wordlist
)


# Replace absent words with zero.
complete_tfidf_table = complete_tfidf_table.fillna(
    0.0
)


print(
    "Vocabulary size:",
    len(wordlist)
)

print(
    "Complete TF-IDF table shape:",
    complete_tfidf_table.shape
)

Vocabulary size: 3127
Complete TF-IDF table shape: (60, 3127)


#### 1.6.7 Creating the Final Matrices and Target Vectors

The tiny metadata is separated into training and test records.

The document IDs determine the row order of each feature matrix. The
ratings are extracted in exactly the same order, ensuring that every
matrix row matches its correct target value.

The four required Task 1 objects are:

- `training_data`
- `test_data`
- `training_targets`
- `test_targets`

In [32]:
# Select tiny training metadata.
tiny_training_metadata = tiny_review_metadata[
    tiny_review_metadata["split"] == "train"
].copy()

tiny_training_metadata = tiny_training_metadata.reset_index(
    drop=True
)


# Select tiny test metadata.
tiny_test_metadata = tiny_review_metadata[
    tiny_review_metadata["split"] == "test"
].copy()

tiny_test_metadata = tiny_test_metadata.reset_index(
    drop=True
)


# Store the required document order.
training_document_ids = tiny_training_metadata[
    "document_id"
].tolist()

test_document_ids = tiny_test_metadata[
    "document_id"
].tolist()


# Create the final training feature matrix.
training_data = complete_tfidf_table.reindex(
    training_document_ids
)


# Create the final test feature matrix.
test_data = complete_tfidf_table.reindex(
    test_document_ids
)


# Create target vectors in the matching row order.
training_targets = tiny_training_metadata[
    "rating"
].to_numpy()

test_targets = tiny_test_metadata[
    "rating"
].to_numpy()

#### 1.6.8 Validating the Final Task 1 Objects

The final matrices and target vectors are checked for:

- correct numbers of training and test reviews;
- equal feature-column counts;
- matching row and target counts;
- missing values; and
- correct row order.

These checks confirm that the data is ready for classification.

In [33]:
print("training_data shape:", training_data.shape)
print("test_data shape:", test_data.shape)

print()
print("training_targets shape:", training_targets.shape)
print("test_targets shape:", test_targets.shape)

print()
print(
    "Same number of feature columns:",
    training_data.shape[1] == test_data.shape[1]
)

print(
    "Training rows match targets:",
    training_data.shape[0] == len(training_targets)
)

print(
    "Test rows match targets:",
    test_data.shape[0] == len(test_targets)
)

print()
print(
    "Missing values in training_data:",
    training_data.isna().sum().sum()
)

print(
    "Missing values in test_data:",
    test_data.isna().sum().sum()
)

print()
print("First 10 training ratings:")
print(training_targets[:10])

print()
print("First 10 test ratings:")
print(test_targets[:10])

training_data shape: (40, 3127)
test_data shape: (20, 3127)

training_targets shape: (40,)
test_targets shape: (20,)

Same number of feature columns: True
Training rows match targets: True
Test rows match targets: True

Missing values in training_data: 0
Missing values in test_data: 0

First 10 training ratings:
[ 9  8 10  7  8  8  7  7  7  7]

First 10 test ratings:
[10  7  9  8  8  9  8  7 10  8]


In [34]:
# Display only a small section because the complete matrix is very wide.
display(
    training_data.iloc[
        0:5,
        0:10
    ]
)

word,a,abandoned,abducted,ability,able,about,above,absence,absolutely,absorb
document_id,,,,,,,,,,
train/pos/0_9.txt,0.030012,0.0,0.0,0.0,0.0,0.011978,0.0,0.0,0.0,0.0
train/pos/10000_8.txt,0.040677,0.0,0.0,0.0,0.0,0.003820,0.0,0.0,0.0,0.0
train/pos/10001_10.txt,0.042017,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
train/pos/10002_7.txt,0.059299,0.0,0.0,0.0,0.0,0.013524,0.0,0.0,0.0,0.0
train/pos/10003_8.txt,0.034725,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


#### 1.6.9 Saving the Prepared Data

The final feature matrices, target vectors and word list are saved in the
local `outputs` folder.

Saving these objects makes the workflow more reliable because they can
be reloaded without rerunning the Hadoop jobs each time the notebook is
opened.

In [35]:
# Import NumPy for saving target vectors.
import numpy as np


# Ensure that the output folder exists.
os.makedirs(
    "outputs",
    exist_ok=True
)


# Save the feature matrices.
training_data.to_pickle(
    "outputs/training_data.pkl"
)

test_data.to_pickle(
    "outputs/test_data.pkl"
)


# Save the target vectors.
np.save(
    "outputs/training_targets.npy",
    training_targets
)

np.save(
    "outputs/test_targets.npy",
    test_targets
)


# Save the word list.
wordlist_table = pd.DataFrame({
    "word": wordlist
})

wordlist_table.to_csv(
    "outputs/wordlist.csv",
    index=False
)


print("Task 1 objects saved successfully.")

Task 1 objects saved successfully.


### Task 1 Conclusion

Task 1 was completed successfully using the 60-review tiny IMDb dataset.

The rating scores were extracted from the review file names using
Python. Hadoop Streaming and two MapReduce jobs were then used to
calculate term frequency, document frequency, inverse document frequency
and final TF-IDF values.

The sparse TF-IDF records were reorganised into two feature matrices:

- `training_data`, containing 40 review rows;
- `test_data`, containing 20 review rows.

Both matrices use the same word list and feature-column order. The
corresponding rating vectors contain 40 training targets and 20 test
targets, with no missing values or alignment errors.

The four required Task 1 objects are therefore ready for sentiment
classification in Task 2.

<hr style="height:4px;border-width:0;color:gray;background-color:green">

## Task 2. Classification (15 points)
Construct a classification model for review sentiment prediction, meaning that given a customer review (taken from test set) about a movie, your program should be able to predict whether it is positive or negative. There is no limitation on how many classifiers and what specific model you should use. You can simply pick one that works for you for this task, either from those covered in lectures and practical classs or any other classifiers from any python packages. A good starting point is the `scikit-learn` (i.e. `sklearn`) package. 
A few things you need to address in your python program are listed as requirements below. 

### Requirements: 

1.	Data pre-processing. In task 1, you have extracted the ratings vectors for training and test. They are raw ratings. As we are interested in sentiment prediction, i.e. to predict either the review is positive or negative. You need to convert all ratings>5 as positive class and ratings<=5 as negative class. Choose a coding scheme, e.g. 1 for positive, 0 for negative. 
2.	Normalisation. Apply at least one normalisation scheme and compare the performance of the classifier(s) with and without normalisation. 
3.	Training and model selection. Use cross validation to select the best parameters for your classifier. There may be many parameters to tune in some classifiers such as random forest classifier (RFC). You can focus on the most important one(s) such as `max_depth` and `n_estimators` in RFC. Refer to `scikit-learn` package documentation for details. 
Hint: you can start with a small subset of training set to test a few parameters to get a feel of what range the parameters should be that make the model perform well in terms of prediction accuracy. Then turn on large scale cross validation on the whole training set.  
4.	Test on test data. After model selection, apply the best model, i.e. model with the parameters that produces the best cross validation scores, to test data and make prediction for each review and record prediction accuracy (ACC). 

Note: 

1. Always train your classifier(s) ONLY on training data including cross validation. After model selection, apply the best model on test data to evaluate the performance. 
2. Good performance, i.e. higher ACC on test data, is not essential for this task. However, if your classifier has ACC low than 60%, it usually means that there are some mistakes somewhere in your code. So try to score as high ACC as possible. 
3.	You are encouraged to try many classifiers. If the coding is right, this should not be too difficult. Remember model selection when you try different classifiers!

### Marking scheme for task 2:

- Data pre-processing (1pts): convert ratings to positive and negative coding scheme. 
- Normalisation and comparison (3pts): apply normalisation and compare performance difference with and without it.
- Training on training data (3pts): training performed on training data.
- Cross validation (6pts): apply cross validation on training data. 
- Testing on test data (2pts): best model applied to test data and ACC produced.  




<hr>

## Task 2: submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

### Solution Approach

Task 1 converted each movie review into a TF-IDF feature vector.

Task 2 uses these feature vectors to build a binary sentiment
classification model. The model predicts whether a review is:

- `1`: positive; or
- `0`: negative.

Only the training data is used for model development, normalisation
comparison and hyperparameter selection. The test data is kept unseen
until the final model has been selected.
Logistic Regression was selected because it is a relatively simple and
interpretable binary classifier that is suitable for high-dimensional TF-IDF
text features. It also supports regularisation, allowing the effect of parameter
`C` to be evaluated through cross-validation.


### 2.1 Converting Ratings into Binary Classes

The target vectors created in Task 1 contain the original IMDb ratings.

For sentiment classification, the ratings are converted using the
assignment rule:

- ratings greater than 5 become positive class `1`;
- ratings equal to or below 5 become negative class `0`.

New variables named `training_labels` and `test_labels` are created so
that the original rating vectors remain unchanged.

In [36]:
# Convert original ratings into binary sentiment classes.
training_labels = np.where(
    training_targets > 5,
    1,
    0
)

test_labels = np.where(
    test_targets > 5,
    1,
    0
)


print("First 10 original training ratings:")
print(training_targets[:10])

print()
print("First 10 binary training labels:")
print(training_labels[:10])

First 10 original training ratings:
[ 9  8 10  7  8  8  7  7  7  7]

First 10 binary training labels:
[1 1 1 1 1 1 1 1 1 1]


#### 2.1.1 Class Distribution Validation

The numbers of positive and negative reviews are counted in both the
training and test sets.

A balanced class distribution is useful because accuracy will not be
dominated by one sentiment class.

In [37]:
# Count training classes.
training_class_counts = pd.Series(
    training_labels
).value_counts().sort_index()

# Count test classes.
test_class_counts = pd.Series(
    test_labels
).value_counts().sort_index()


class_distribution = pd.DataFrame({
    "Class": ["Negative (0)", "Positive (1)"],
    "Training reviews": [
        training_class_counts.get(0, 0),
        training_class_counts.get(1, 0)
    ],
    "Test reviews": [
        test_class_counts.get(0, 0),
        test_class_counts.get(1, 0)
    ]
})


display(class_distribution)

,Class,Training reviews,Test reviews
0,Negative (0),20,10
1,Positive (1),20,10


#### Interpretation

The original ratings were converted successfully into binary sentiment
classes.

The training set contains 20 negative and 20 positive reviews. The test
set contains 10 negative and 10 positive reviews. Both datasets are
therefore balanced, so accuracy is an appropriate primary evaluation
metric for this experiment.

### 2.2 Cross-Validation and Normalisation Comparison

TF-IDF review vectors can have different Euclidean norms even though
they have the same number of feature dimensions.

L2 normalisation rescales each review vector so that its Euclidean norm
equals one while preserving the relative pattern of its word weights.

Two Logistic Regression pipelines are compared:

1. Logistic Regression without additional normalisation;
2. L2 normalisation followed by Logistic Regression.

Both pipelines are evaluated using the same five-fold stratified
cross-validation procedure.

Stratification keeps similar positive and negative class proportions in
every fold. The test data is not used in this comparison.

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer


# Use five reproducible stratified folds.
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


print("Cross-validation folds:", cross_validation.get_n_splits())

Cross-validation folds: 5


In [39]:
# Model without additional normalisation.
model_without_normalisation = Pipeline([
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])


# Model with L2 normalisation.
model_with_normalisation = Pipeline([
    (
        "normaliser",
        Normalizer(norm="l2")
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

In [40]:
# Evaluate the model without normalisation.
scores_without_normalisation = cross_val_score(
    model_without_normalisation,
    training_data,
    training_labels,
    cv=cross_validation,
    scoring="accuracy"
)


# Evaluate the model with L2 normalisation.
scores_with_normalisation = cross_val_score(
    model_with_normalisation,
    training_data,
    training_labels,
    cv=cross_validation,
    scoring="accuracy"
)


normalisation_comparison = pd.DataFrame({
    "Method": [
        "Without normalisation",
        "L2 normalisation"
    ],
    "Fold 1": [
        scores_without_normalisation[0],
        scores_with_normalisation[0]
    ],
    "Fold 2": [
        scores_without_normalisation[1],
        scores_with_normalisation[1]
    ],
    "Fold 3": [
        scores_without_normalisation[2],
        scores_with_normalisation[2]
    ],
    "Fold 4": [
        scores_without_normalisation[3],
        scores_with_normalisation[3]
    ],
    "Fold 5": [
        scores_without_normalisation[4],
        scores_with_normalisation[4]
    ],
    "Mean accuracy": [
        scores_without_normalisation.mean(),
        scores_with_normalisation.mean()
    ],
    "Standard deviation": [
        scores_without_normalisation.std(),
        scores_with_normalisation.std()
    ]
})


display(
    normalisation_comparison.round(4)
)

,Method,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean accuracy,Standard deviation
0,Without normalisation,0.75,0.625,0.625,0.750,0.625,0.675,0.0612
1,L2 normalisation,0.50,0.750,1.000,0.875,0.750,0.775,0.1658


In [41]:
mean_without_normalisation = (
    scores_without_normalisation.mean()
)

mean_with_normalisation = (
    scores_with_normalisation.mean()
)


if mean_with_normalisation > mean_without_normalisation:

    better_method = "L2 normalisation"

elif mean_with_normalisation < mean_without_normalisation:

    better_method = "No additional normalisation"

else:

    better_method = "Both methods produced the same mean accuracy"


print("Mean accuracy without normalisation:")
print(round(mean_without_normalisation, 4))

print()
print("Mean accuracy with L2 normalisation:")
print(round(mean_with_normalisation, 4))

print()
print("Better result:")
print(better_method)

Mean accuracy without normalisation:
0.675

Mean accuracy with L2 normalisation:
0.775

Better result:
L2 normalisation


#### Interpretation

L2 normalisation produced a higher mean cross-validation accuracy than
the model without additional normalisation.

The model without normalisation achieved a mean accuracy of 0.675, while
L2 normalisation achieved 0.775. This represents an improvement of 0.10,
or 10 percentage points.

However, the L2-normalised model had a cross-validation standard
deviation of approximately 0.166, compared with 0.061 for the
unnormalised model. Its performance therefore varied more strongly
between folds.

This instability is likely related to the very small training sample of
40 reviews. The normalisation method and regularisation parameter are
selected together in the following GridSearchCV step.

### 2.3 Hyperparameter and Model Selection

Logistic Regression uses the parameter `C` to control regularisation.

- a smaller `C` applies stronger regularisation;
- a larger `C` allows the model to fit the training data more closely.

Very weak regularisation can cause overfitting, while very strong
regularisation can cause underfitting.

GridSearchCV evaluates several `C` values and both preprocessing
approaches using five-fold stratified cross-validation. This allows the
normalisation method and model parameter to be selected together without
using the test data.

In [42]:
from sklearn.model_selection import GridSearchCV


# Create one pipeline for the full search.
search_pipeline = Pipeline([
    (
        "normaliser",
        "passthrough"
    ),
    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])


# Test both normalisation choices and several C values.
parameter_grid = {
    "normaliser": [
        "passthrough",
        Normalizer(norm="l2")
    ],
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ]
}

In [43]:
grid_search = GridSearchCV(
    estimator=search_pipeline,
    param_grid=parameter_grid,
    scoring="accuracy",
    cv=cross_validation,
    refit=True,
    return_train_score=True
)


# Fit only on the training data.
grid_search.fit(
    training_data,
    training_labels
)


print("Best cross-validation accuracy:")
print(round(grid_search.best_score_, 4))

print()
print("Best parameters:")
print(grid_search.best_params_)

Best cross-validation accuracy:
0.775

Best parameters:
{'classifier__C': 1.0, 'normaliser': Normalizer()}


In [44]:
grid_search_results = pd.DataFrame(
    grid_search.cv_results_
)


grid_search_summary = grid_search_results[[
    "param_normaliser",
    "param_classifier__C",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].copy()


grid_search_summary = grid_search_summary.sort_values(
    by="rank_test_score"
).reset_index(drop=True)


grid_search_summary.columns = [
    "Normalisation",
    "C",
    "Mean training accuracy",
    "Mean CV accuracy",
    "CV standard deviation",
    "Rank"
]


display(
    grid_search_summary.round(4)
)

,Normalisation,C,Mean training accuracy,Mean CV accuracy,CV standard deviation,Rank
0,Normalizer(),1.00,1.0,0.775,0.1658,1
1,Normalizer(),10.00,1.0,0.750,0.1768,2
2,Normalizer(),100.00,1.0,0.750,0.1768,2
3,passthrough,100.00,1.0,0.750,0.1118,2
4,passthrough,10.00,1.0,0.700,0.1000,5
5,Normalizer(),0.10,1.0,0.675,0.0612,6
6,passthrough,0.10,1.0,0.675,0.0612,6
7,passthrough,0.01,1.0,0.675,0.0612,6
8,passthrough,1.00,1.0,0.675,0.0612,6
9,Normalizer(),0.01,1.0,0.575,0.0612,10


#### Interpretation

GridSearchCV selected L2 normalisation with a Logistic Regression
regularisation parameter of `C = 1.0`.

This configuration achieved the highest mean cross-validation accuracy
of 0.775. The result is consistent with the earlier normalisation
comparison, in which L2 normalisation also outperformed the model without
additional normalisation.

All evaluated configurations achieved a mean training accuracy of 1.00,
while their cross-validation accuracies were lower. This difference
suggests that the models fitted the small training sample very strongly.

The training matrix contains only 40 reviews but more than 3,000 TF-IDF
features. In such a high-dimensional setting, perfect training
classification can occur relatively easily. Cross-validation is
therefore more informative than training accuracy when selecting the
final model.

### 2.4 Final Evaluation on the Test Data

After model selection, the best pipeline is applied to the untouched test
data.

This is the first time that test features are used in Task 2. The test
set therefore provides an independent estimate of the selected model's
performance.

The following evaluation reports:

- prediction accuracy;
- a confusion matrix;
- precision, recall and F1-score; and
- individual predictions for the 20 test reviews.

In [45]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


# GridSearchCV already refitted the best model.
best_model = grid_search.best_estimator_


# Predict the unseen test reviews.
test_predictions = best_model.predict(
    test_data
)


# Calculate final test accuracy.
test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)


print("Final test accuracy:")
print(round(test_accuracy, 4))

print()
print("Correct predictions:")
print(
    int(
        (test_predictions == test_labels).sum()
    ),
    "out of",
    len(test_labels)
)

Final test accuracy:
0.8

Correct predictions:
16 out of 20


In [46]:
final_confusion_matrix = confusion_matrix(
    test_labels,
    test_predictions
)


confusion_matrix_table = pd.DataFrame(
    final_confusion_matrix,
    index=[
        "Actual negative",
        "Actual positive"
    ],
    columns=[
        "Predicted negative",
        "Predicted positive"
    ]
)


display(confusion_matrix_table)


print("Classification report:")
print()

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "Negative",
            "Positive"
        ],
        zero_division=0
    )
)

,Predicted negative,Predicted positive
Actual negative,8,2
Actual positive,2,8


Classification report:

              precision    recall  f1-score   support

    Negative       0.80      0.80      0.80        10
    Positive       0.80      0.80      0.80        10

    accuracy                           0.80        20
   macro avg       0.80      0.80      0.80        20
weighted avg       0.80      0.80      0.80        20



In [47]:
prediction_results = tiny_test_metadata[[
    "document_id",
    "rating"
]].copy()


prediction_results["actual_class"] = test_labels

prediction_results["predicted_class"] = (
    test_predictions
)

prediction_results["correct_prediction"] = (
    prediction_results["actual_class"]
    == prediction_results["predicted_class"]
)


display(prediction_results)

,document_id,rating,actual_class,predicted_class,correct_prediction
0,test/pos/0_10.txt,10,1,0,False
1,test/pos/10000_7.txt,7,1,1,True
2,test/pos/10001_9.txt,9,1,1,True
3,test/pos/10002_8.txt,8,1,1,True
4,test/pos/10003_8.txt,8,1,1,True
5,test/pos/10004_9.txt,9,1,1,True
6,test/pos/10005_8.txt,8,1,1,True
7,test/pos/10006_7.txt,7,1,1,True
8,test/pos/10007_10.txt,10,1,0,False
9,test/pos/10008_8.txt,8,1,1,True


In [48]:
incorrect_predictions = prediction_results[
    prediction_results["correct_prediction"] == False
]


print(
    "Incorrect predictions:",
    len(incorrect_predictions)
)


if len(incorrect_predictions) > 0:

    display(
        incorrect_predictions
    )

else:

    print(
        "All test reviews were classified correctly."
    )

Incorrect predictions: 4


,document_id,rating,actual_class,predicted_class,correct_prediction
0,test/pos/0_10.txt,10,1,0,False
8,test/pos/10007_10.txt,10,1,0,False
15,test/neg/10004_2.txt,2,0,1,False
16,test/neg/10005_2.txt,2,0,1,False


#### Interpretation

The selected model correctly classified 16 of the 20 unseen test
reviews, producing a final test accuracy of 0.80.

The confusion matrix was balanced:

- 8 of 10 negative reviews were classified correctly;
- 8 of 10 positive reviews were classified correctly;
- 2 negative reviews were incorrectly classified as positive; and
- 2 positive reviews were incorrectly classified as negative.

Precision, recall and F1-score were all 0.80 for both sentiment classes.
The model therefore did not show a clear bias towards either positive or
negative predictions.

The final test accuracy was slightly higher than the best
cross-validation accuracy of 0.775. However, the test set contains only
20 reviews, meaning that each prediction changes accuracy by five
percentage points. The result should therefore be interpreted as evidence
that the pipeline works correctly, rather than as a stable estimate of
performance on the complete IMDb dataset.

### 2.5 Saving the Final Classification Results

The selected model configuration, cross-validation results and final
test predictions are saved in the `outputs` folder.

This preserves the results and makes them available without rerunning
the complete model-selection process.

In [49]:
import joblib


# Save the selected model.
joblib.dump(
    best_model,
    "outputs/best_sentiment_model.joblib"
)


# Save the complete GridSearchCV results.
grid_search_summary.to_csv(
    "outputs/grid_search_results.csv",
    index=False
)


# Save individual test predictions.
prediction_results.to_csv(
    "outputs/test_predictions.csv",
    index=False
)


# Save a simple performance summary.
performance_summary = pd.DataFrame({
    "Metric": [
        "Best cross-validation accuracy",
        "Final test accuracy"
    ],
    "Value": [
        grid_search.best_score_,
        test_accuracy
    ]
})


performance_summary.to_csv(
    "outputs/classification_performance.csv",
    index=False
)


print("Task 2 results saved successfully.")

Task 2 results saved successfully.


### Task 2 Conclusion

Task 2 successfully converted the original IMDb ratings into binary
sentiment classes, where positive reviews were coded as `1` and negative
reviews as `0`.

Logistic Regression was evaluated with and without L2 normalisation using
five-fold stratified cross-validation. L2 normalisation improved mean
cross-validation accuracy from 0.675 to 0.775.

GridSearchCV evaluated two preprocessing choices and five regularisation
values. It selected L2 normalisation with `C = 1.0` as the best
configuration. Model selection was conducted using only the training
data.

The selected pipeline was then evaluated once on the untouched test set.
It correctly classified 16 of 20 reviews and achieved an accuracy of
0.80. Precision, recall and F1-score were also 0.80 for both sentiment
classes.

The results demonstrate a complete and correctly separated
training-validation-testing workflow. Nevertheless, the perfect training
accuracy and variation between cross-validation folds indicate that the
results are influenced by the small 60-review experimental dataset.

<hr style="height:4px;border-width:0;color:gray;background-color:blue">

## Bonus Task (10 points)
This is a bonus task. It is not essential but if you could complete it as required you will receive 10 extra points towards your final results of this unit. The task is similar to the _Task 5_ in prac 8. 

Compute the correlation between features and response. Use the TF-IDF as features and review scores as response. Consider only training set, i.e. on `training_data`. Here is the details. Let $\mathbf x_i$ be the $i$-th TF-IDF _column_ vector you extracted for the $i$-th review, and $y_i$ its corresponding review score. To compute the coorelation, we need $\tilde{\mathbf x}_i$ and $\tilde{y}_i$, normalised version of $\mathbf x_i$ and $y_i$ as the following. 
$$
\tilde{\mathbf x}_i = \frac{\hat{\mathbf x}_i}{\|\hat{\mathbf x}_i\|} 
$$
where $\hat{\mathbf x}_i = \mathbf x_i - \mathbf m$, $\mathbf m$ is the mean of all features, i.e. $ \mathbf m = \frac{\sum_{i=1}^N \mathbf x_i}N$, and $\|\hat{\mathbf x}_i\|$ is the so-called $\ell_2$ norm of $\|\hat{\mathbf x}_i\|$ which is defined as 
$$\|\hat{\mathbf x}_i\| = \sqrt{\sum_{j=1}^Dx_{i_j}^2}$$
i.e. the square root of the sum of squares of all the elements in vector $\hat{\mathbf x}_i$. $\tilde{y}_i$ is similar 
$$ \tilde{y}_i = \frac{y_i}{\|\mathbf y\|}$$
where $\mathbf y=[y_1,\ldots,y_N]$ is the vector of all review scores. Then the correlation $\mathbf r$ is  
$$
\mathbf r = \sum_{i=1}^N\tilde{y}_i\tilde{\mathbf x}_i.
$$
$\mathbf r$ will be a vector of length $D$. 

### Requirements: 

1. Use map reduce computing model for this task is mandatory. Direct computing the correlation from the matrices obtained from Task 1, i.e. `training_data` is _not_ acceptable. 

2. Python code and Hadoop streaming commands must be supplied for this taks. If multiple map reduce steps are used, a step-by-step guidance must be provided as well. 


_Hint: you may consider several map reduce to compute mean, $\ell_2$ norm, multiplication and etc._

<hr>

## Bonus task : submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

### Bonus Solution

#### B.1 Correlation Method

This bonus task computes the correlation between the TF-IDF features and
the original review ratings using only the training set.

The correlation is calculated using the normalisation procedure specified
in the assignment.

For review $i$, let:

- $\mathbf{x}_i$ be its TF-IDF feature vector;
- $y_i$ be its original numerical rating; and
- $\mathbf{m}$ be the mean TF-IDF vector across all training reviews.

The centred and normalised feature vector is:

$$
\tilde{\mathbf{x}}_i
=
\frac{\mathbf{x}_i-\mathbf{m}}
{\left\|\mathbf{x}_i-\mathbf{m}\right\|_2}
$$

The rating is normalised by the L2 norm of the complete rating vector:

$$
\tilde{y}_i
=
\frac{y_i}
{\left\|\mathbf{y}\right\|_2}
$$

where:

$$
\left\|\mathbf{y}\right\|_2
=
\sqrt{
\sum_{i=1}^{N} y_i^2
}
$$

Unlike the TF-IDF feature vectors, the rating vector is not centred,
following the formula specified in the assignment.

The final correlation vector is:

$$
\mathbf{r}
=
\sum_{i=1}^{N}
\tilde{y}_i\tilde{\mathbf{x}}_i
$$

The result contains one correlation value for every word in the TF-IDF
vocabulary.

The calculation is performed from the sparse Hadoop TF-IDF output. The
`training_data` matrix is not used to calculate the correlation.

#### B.2 Preparing the Training Rating Input

The bonus calculation uses the 40 reviews in the tiny training set.

A small tab-separated file is created containing one document ID and its
original numerical rating per row. This file is used as the input for the
first bonus MapReduce job.

In [50]:
# Select the document IDs and original ratings.
bonus_training_ratings = tiny_training_metadata[[
    "document_id",
    "rating"
]].copy()


# Sort the records to make the output reproducible.
bonus_training_ratings = bonus_training_ratings.sort_values(
    by="document_id"
).reset_index(drop=True)


# Save the input without column names.
bonus_training_ratings.to_csv(
    "outputs/bonus_training_ratings.tsv",
    sep="\t",
    header=False,
    index=False
)


print(
    "Training rating records:",
    len(bonus_training_ratings)
)

print(
    "Unique training documents:",
    bonus_training_ratings["document_id"].nunique()
)

print(
    "Rating range:",
    bonus_training_ratings["rating"].min(),
    "to",
    bonus_training_ratings["rating"].max()
)

display(
    bonus_training_ratings.head()
)

Training rating records: 40
Unique training documents: 40
Rating range: 1 to 10


,document_id,rating
0,train/neg/0_3.txt,3
1,train/neg/10000_4.txt,4
2,train/neg/10001_4.txt,4
3,train/neg/10002_1.txt,1
4,train/neg/10003_1.txt,1


#### B.3 MapReduce Job 1: Rating Statistics

The first bonus MapReduce job calculates the statistics required to
normalise the rating vector.

The mapper emits the following values for every training review:

- one document count;
- the rating; and
- the squared rating.

The reducer calculates:

- the number of training reviews;
- the sum of ratings;
- the sum of squared ratings;
- the mean rating for descriptive reporting; and
- the L2 norm of the original rating vector.

The rating-vector norm follows the assignment formula:

$$
\left\|\mathbf{y}\right\|_2
=
\sqrt{
\sum_{i=1}^{N} y_i^2
}
$$

In [51]:
%%writefile bonus_rating_mapper.py
#!/usr/bin/env python3

import sys


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 2:
        continue

    document_id = fields[0]

    try:
        rating = float(fields[1])

    except ValueError:
        continue

    rating_squared = rating * rating

    print(
        "stats"
        + "\t"
        + "1"
        + "\t"
        + str(rating)
        + "\t"
        + str(rating_squared)
    )

Overwriting bonus_rating_mapper.py


In [52]:
%%writefile bonus_rating_reducer.py
#!/usr/bin/env python3

import math
import sys


number_of_documents = 0
sum_of_ratings = 0.0
sum_of_squared_ratings = 0.0


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 4:
        continue

    number_of_documents += int(fields[1])
    sum_of_ratings += float(fields[2])
    sum_of_squared_ratings += float(fields[3])


if number_of_documents > 0:

    # Keep the mean for descriptive reporting.
    mean_rating = (
        sum_of_ratings
        / number_of_documents
    )

    # Calculate the L2 norm required by the assignment.
    # The rating vector is not centred.
    rating_norm = math.sqrt(
        sum_of_squared_ratings
    )

    print(
        "stats"
        + "\t"
        + str(number_of_documents)
        + "\t"
        + format(sum_of_ratings, ".6f")
        + "\t"
        + format(sum_of_squared_ratings, ".6f")
        + "\t"
        + format(mean_rating, ".6f")
        + "\t"
        + format(rating_norm, ".6f")
    )

Overwriting bonus_rating_reducer.py


##### Local Validation

A three-review sample is used to confirm that the mapper and reducer
correctly calculate the response statistics before Hadoop execution.

In [53]:
%%bash

cat > sample_bonus_ratings.tsv << 'EOF'
train/pos/document1_9.txt	9
train/neg/document2_2.txt	2
train/pos/document3_8.txt	8
EOF


cat sample_bonus_ratings.tsv \
    | python3 bonus_rating_mapper.py \
    | sort \
    | python3 bonus_rating_reducer.py


rm sample_bonus_ratings.tsv

stats	3	19.000000	149.000000	6.333333	12.206556


In [54]:
%%bash

LOCAL_INPUT="outputs/bonus_training_ratings.tsv"

HDFS_INPUT="/users/hadoop/COMP3002/bonus_training_ratings"

HDFS_OUTPUT="/users/hadoop/COMP3002/bonus_rating_stats"


STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


# Recreate the HDFS input folder.
hdfs dfs -rm -r -f "${HDFS_INPUT}"
hdfs dfs -mkdir -p "${HDFS_INPUT}"

hdfs dfs -put \
    "${LOCAL_INPUT}" \
    "${HDFS_INPUT}/"


# Remove the previous result.
hdfs dfs -rm -r -f "${HDFS_OUTPUT}"


# Run Bonus MapReduce Job 1.
hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.job.reduces=1 \
    -files bonus_rating_mapper.py,bonus_rating_reducer.py \
    -mapper "python3 bonus_rating_mapper.py" \
    -reducer "python3 bonus_rating_reducer.py" \
    -input "${HDFS_INPUT}" \
    -output "${HDFS_OUTPUT}"

Deleted /users/hadoop/COMP3002/bonus_training_ratings
Deleted /users/hadoop/COMP3002/bonus_rating_stats
packageJobJar: [/tmp/hadoop-unjar2972229844914196274/] [] /tmp/streamjob2398160384058793944.jar tmpDir=null


2026-07-17 20:42:26,390 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:42:26,722 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:42:27,050 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0013
2026-07-17 20:42:27,714 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-17 20:42:27,876 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-17 20:42:28,089 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0013
2026-07-17 20:42:28,091 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:42:28,383 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:42:28,405 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:42:28,518 INFO impl.YarnClientImpl: Submitted application application_17842624431

In [55]:
%%bash

HDFS_OUTPUT="/users/hadoop/COMP3002/bonus_rating_stats"

LOCAL_OUTPUT="outputs/bonus_rating_stats.tsv"


rm -f "${LOCAL_OUTPUT}"

hdfs dfs -getmerge \
    "${HDFS_OUTPUT}" \
    "${LOCAL_OUTPUT}"


echo "Rating statistics:"
cat "${LOCAL_OUTPUT}"

Rating statistics:
stats	40	210.000000	1438.000000	5.250000	37.920970


#### B.4 MapReduce Job 2: Mean TF-IDF Vector

The second bonus MapReduce job calculates the mean value of every TF-IDF
feature across the 40 training reviews.

For word $j$, the mean TF-IDF value is:

$$
m_j
=
\frac{1}{N}
\sum_{i=1}^{N} x_{ij}
$$

where:

- $m_j$ is the mean TF-IDF value of feature $j$;
- $N$ is the number of training reviews; and
- $x_{ij}$ is the TF-IDF value of feature $j$ in review $i$.

Words absent from a review have a TF-IDF value of zero. Because zero
values do not affect the sum, the sparse Hadoop output can be used
directly without creating every missing document-word combination.

Records from the test set are emitted with a value of zero. This keeps
words that occur only in the test set within the complete vocabulary,
while ensuring that only training TF-IDF values contribute to the mean.

In [56]:
%%writefile bonus_mean_mapper.py
#!/usr/bin/env python3

import sys


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 7:
        continue

    document_id = fields[0]
    word = fields[1]

    try:
        tfidf = float(fields[6])

    except ValueError:
        continue

    # Only training TF-IDF values contribute to the mean.
    if document_id.startswith("train/"):

        print(
            word
            + "\t"
            + str(tfidf)
        )

    else:

        # Preserve test-only vocabulary items.
        print(
            word
            + "\t"
            + "0"
        )

Overwriting bonus_mean_mapper.py


In [57]:
%%writefile bonus_mean_reducer.py
#!/usr/bin/env python3

import os
import sys


total_documents_text = os.environ.get(
    "TOTAL_TRAINING_DOCUMENTS"
)

if total_documents_text is None:
    total_documents = 40

else:
    total_documents = int(
        total_documents_text
    )


current_word = None
current_sum = 0.0


def print_mean(word, feature_sum):

    feature_mean = (
        feature_sum
        / total_documents
    )

    print(
        word
        + "\t"
        + format(feature_mean, ".10f")
    )


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 2:
        continue

    word = fields[0]

    try:
        tfidf = float(fields[1])

    except ValueError:
        continue

    if word == current_word:

        current_sum += tfidf

    else:

        if current_word is not None:

            print_mean(
                current_word,
                current_sum
            )

        current_word = word
        current_sum = tfidf


if current_word is not None:

    print_mean(
        current_word,
        current_sum
    )

Overwriting bonus_mean_reducer.py


In [58]:
%%bash

INPUT_PATH="/users/hadoop/COMP3002/job2_tfidf"

OUTPUT_PATH="/users/hadoop/COMP3002/bonus_feature_means"


STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


hdfs dfs -rm -r -f "${OUTPUT_PATH}"


hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.job.reduces=1 \
    -files bonus_mean_mapper.py,bonus_mean_reducer.py \
    -cmdenv TOTAL_TRAINING_DOCUMENTS=40 \
    -mapper "python3 bonus_mean_mapper.py" \
    -reducer "python3 bonus_mean_reducer.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}"

Deleted /users/hadoop/COMP3002/bonus_feature_means
packageJobJar: [/tmp/hadoop-unjar8190923250682978779/] [] /tmp/streamjob3890433422850543589.jar tmpDir=null


2026-07-17 20:42:55,640 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:42:55,871 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:42:56,122 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0014
2026-07-17 20:42:56,744 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-17 20:42:56,915 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-17 20:42:57,118 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0014
2026-07-17 20:42:57,132 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:42:57,383 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:42:57,387 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:42:57,478 INFO impl.YarnClientImpl: Submitted application application_17842624431

In [59]:
%%bash

HDFS_OUTPUT="/users/hadoop/COMP3002/bonus_feature_means"

LOCAL_OUTPUT="outputs/bonus_feature_means.tsv"


rm -f "${LOCAL_OUTPUT}"

hdfs dfs -getmerge \
    "${HDFS_OUTPUT}" \
    "${LOCAL_OUTPUT}"


echo "Number of feature means:"

wc -l < "${LOCAL_OUTPUT}"


echo
echo "First 10 feature means:"

awk 'NR <= 10 {print}' "${LOCAL_OUTPUT}"

Number of feature means:
3127

First 10 feature means:
a	0.0313682500
abandoned	0.0003903000
abducted	0.0002516000
ability	0.0002754500
able	0.0005764500
about	0.0052846250
above	0.0012330250
absence	0.0007218500
absolutely	0.0007314750
absorb	0.0000000000


#### B.5 Preparing Normalisation Scalars

The feature means and rating statistics produced by MapReduce are loaded
into the local Python environment.

The following feature scalar is required:

$$
\sum_{j=1}^{D} m_j^2
$$

where:

- $D$ is the number of TF-IDF features; and
- $m_j$ is the mean value of feature $j$ across the training reviews.

This value is calculated from the feature-mean output produced by
MapReduce.

The rating mean is retained for descriptive reporting only. Rating
normalisation uses the uncentred L2 norm $\|\mathbf{y}\|_2$, as
specified in the assignment.

These values allow the centred L2 norm of each review to be calculated
from sparse TF-IDF records without constructing a complete dense matrix.

In [60]:
# Load rating statistics produced by MapReduce.
bonus_rating_stats = pd.read_csv(
    "outputs/bonus_rating_stats.tsv",
    sep="\t",
    header=None,
    names=[
        "key",
        "number_of_documents",
        "sum_of_ratings",
        "sum_of_squared_ratings",
        "mean_rating",
        "rating_norm"
    ]
)


# Load feature means produced by MapReduce.
bonus_feature_means = pd.read_csv(
    "outputs/bonus_feature_means.tsv",
    sep="\t",
    header=None,
    names=[
        "word",
        "feature_mean"
    ]
)


number_of_training_documents = int(
    bonus_rating_stats.loc[
        0,
        "number_of_documents"
    ]
)

mean_rating = float(
    bonus_rating_stats.loc[
        0,
        "mean_rating"
    ]
)

rating_norm = float(
    bonus_rating_stats.loc[
        0,
        "rating_norm"
    ]
)


feature_mean_squared_sum = (
    bonus_feature_means["feature_mean"]
    .pow(2)
    .sum()
)


bonus_normalisation_scalars = pd.DataFrame({
    "name": [
        "number_of_training_documents",
        "mean_rating",
        "rating_norm",
        "feature_mean_squared_sum"
    ],
    "value": [
        number_of_training_documents,
        mean_rating,
        rating_norm,
        feature_mean_squared_sum
    ]
})


bonus_normalisation_scalars.to_csv(
    "outputs/bonus_normalisation_scalars.tsv",
    sep="\t",
    header=False,
    index=False
)


display(
    bonus_normalisation_scalars
)

print()
print(
    "Number of feature means:",
    len(bonus_feature_means)
)

,name,value
0,number_of_training_documents,40.00000
1,mean_rating,5.25000
2,rating_norm,37.92097
3,feature_mean_squared_sum,0.01404



Number of feature means: 3127


#### B.6 MapReduce Job 3: Centred Review Norms

The third bonus MapReduce job calculates the L2 norm of every centred
training review.

Creating every zero-valued document-word combination would make the data
dense. Instead, the following algebraic identity is used:

$$
\left\|\mathbf{x}_i-\mathbf{m}\right\|_2^2
=
\sum_{j=1}^{D} x_{ij}^2
-
2\sum_{j=1}^{D} x_{ij}m_j
+
\sum_{j=1}^{D} m_j^2
$$

where:

- $\mathbf{x}_i$ is the TF-IDF vector of review $i$;
- $\mathbf{m}$ is the mean TF-IDF vector;
- $x_{ij}$ is the TF-IDF value of feature $j$ in review $i$; and
- $D$ is the number of features.

This identity allows the centred vector norm to be calculated from sparse
TF-IDF records without explicitly creating all zero-valued features.

The mapper reads the sparse TF-IDF records and joins each word with its
feature mean.

The reducer calculates:

- the centred TF-IDF norm of each training review;
- the rating normalised by the L2 norm of the complete rating vector; and
- the baseline contribution required by the final correlation job.

In [61]:
%%writefile bonus_document_mapper.py
#!/usr/bin/env python3

import sys


# Load the feature means distributed by Hadoop.
feature_means = {}

with open(
    "bonus_feature_means.tsv",
    "r",
    encoding="utf-8"
) as mean_file:

    for line in mean_file:

        fields = line.strip().split("\t")

        if len(fields) != 2:
            continue

        feature_means[fields[0]] = float(
            fields[1]
        )


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 7:
        continue

    document_id = fields[0]
    word = fields[1]

    # The bonus uses training data only.
    if not document_id.startswith("train/"):
        continue

    try:
        tfidf = float(fields[6])

    except ValueError:
        continue

    feature_mean = feature_means.get(
        word,
        0.0
    )

    print(
        document_id
        + "\t"
        + word
        + "\t"
        + str(tfidf)
        + "\t"
        + str(feature_mean)
    )

Overwriting bonus_document_mapper.py


In [62]:
%%writefile bonus_document_reducer.py
#!/usr/bin/env python3

import math
import os
import sys


rating_norm = float(
    os.environ.get(
        "RATING_NORM",
        "1"
    )
)

feature_mean_squared_sum = float(
    os.environ.get(
        "FEATURE_MEAN_SQUARED_SUM",
        "0"
    )
)


def extract_rating(document_id):

    file_name = document_id.split("/")[-1]

    rating_text = (
        file_name
        .split("_")[1]
        .split(".")[0]
    )

    return float(rating_text)


def print_document_statistics(
    document_id,
    sum_x_squared,
    sum_x_times_mean
):

    centred_norm_squared = (
        sum_x_squared
        - (2.0 * sum_x_times_mean)
        + feature_mean_squared_sum
    )

    centred_norm_squared = max(
        centred_norm_squared,
        0.0
    )

    centred_feature_norm = math.sqrt(
        centred_norm_squared
    )

    rating = extract_rating(
        document_id
    )

    if rating_norm > 0:

        normalised_rating = (
            rating
            / rating_norm
        )

    else:

        normalised_rating = 0.0


    if centred_feature_norm > 0:

        baseline_component = (
            normalised_rating
            / centred_feature_norm
        )

    else:

        baseline_component = 0.0


    print(
        document_id
        + "\t"
        + format(rating, ".6f")
        + "\t"
        + format(centred_feature_norm, ".10f")
        + "\t"
        + format(normalised_rating, ".10f")
        + "\t"
        + format(baseline_component, ".10f")
    )


current_document = None
current_sum_x_squared = 0.0
current_sum_x_times_mean = 0.0


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 4:
        continue

    document_id = fields[0]

    try:
        tfidf = float(fields[2])
        feature_mean = float(fields[3])

    except ValueError:
        continue

    if document_id == current_document:

        current_sum_x_squared += (
            tfidf * tfidf
        )

        current_sum_x_times_mean += (
            tfidf * feature_mean
        )

    else:

        if current_document is not None:

            print_document_statistics(
                current_document,
                current_sum_x_squared,
                current_sum_x_times_mean
            )

        current_document = document_id

        current_sum_x_squared = (
            tfidf * tfidf
        )

        current_sum_x_times_mean = (
            tfidf * feature_mean
        )


if current_document is not None:

    print_document_statistics(
        current_document,
        current_sum_x_squared,
        current_sum_x_times_mean
    )

Overwriting bonus_document_reducer.py


In [63]:
%%bash

INPUT_PATH="/users/hadoop/COMP3002/job2_tfidf"

OUTPUT_PATH="/users/hadoop/COMP3002/bonus_document_stats"


RATING_NORM=$(awk -F '\t' \
    '$1 == "rating_norm" {print $2}' \
    outputs/bonus_normalisation_scalars.tsv)

FEATURE_MEAN_SQUARED_SUM=$(awk -F '\t' \
    '$1 == "feature_mean_squared_sum" {print $2}' \
    outputs/bonus_normalisation_scalars.tsv)


STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


hdfs dfs -rm -r -f "${OUTPUT_PATH}"


hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.job.reduces=1 \
    -files bonus_document_mapper.py,bonus_document_reducer.py,outputs/bonus_feature_means.tsv#bonus_feature_means.tsv \
    -cmdenv RATING_NORM="${RATING_NORM}" \
    -cmdenv FEATURE_MEAN_SQUARED_SUM="${FEATURE_MEAN_SQUARED_SUM}" \
    -mapper "python3 bonus_document_mapper.py" \
    -reducer "python3 bonus_document_reducer.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}" 

Deleted /users/hadoop/COMP3002/bonus_document_stats
packageJobJar: [/tmp/hadoop-unjar8351634792332345724/] [] /tmp/streamjob8856190558895992677.jar tmpDir=null


2026-07-17 20:43:25,055 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:43:25,324 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:43:25,585 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0015
2026-07-17 20:43:26,333 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-17 20:43:26,545 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-17 20:43:26,762 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0015
2026-07-17 20:43:26,762 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:43:27,331 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:43:27,333 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:43:27,508 INFO impl.YarnClientImpl: Submitted application application_17842624431

In [64]:
%%bash

HDFS_OUTPUT="/users/hadoop/COMP3002/bonus_document_stats"

LOCAL_OUTPUT="outputs/bonus_document_stats.tsv"


rm -f "${LOCAL_OUTPUT}"

hdfs dfs -getmerge \
    "${HDFS_OUTPUT}" \
    "${LOCAL_OUTPUT}"


echo "Number of training document records:"

wc -l < "${LOCAL_OUTPUT}"


echo
echo "First 10 document statistics:"

awk 'NR <= 10 {print}' "${LOCAL_OUTPUT}"

Number of training document records:
40

First 10 document statistics:
train/neg/0_3.txt	3.000000	0.2882058165	0.0791119004	0.2744979311
train/neg/10000_4.txt	4.000000	0.1508898691	0.1054825338	0.6990696888
train/neg/10001_4.txt	4.000000	0.2327396760	0.1054825338	0.4532211079
train/neg/10002_1.txt	1.000000	0.2302628951	0.0263706335	0.1145240245
train/neg/10003_1.txt	1.000000	0.1613950944	0.0263706335	0.1633917905
train/neg/10004_3.txt	3.000000	0.3262935113	0.0791119004	0.2424562476
train/neg/10005_3.txt	3.000000	0.1806368550	0.0791119004	0.4379610150
train/neg/10006_4.txt	4.000000	0.2272945151	0.1054825338	0.4640786592
train/neg/10007_1.txt	1.000000	0.3692991170	0.0263706335	0.0714072475
train/neg/10008_2.txt	2.000000	0.1820250419	0.0527412669	0.2897473136


#### B.7 Calculating the Baseline Sum

For feature $j$, the final correlation can be written as:

$$
r_j
=
\sum_{i:x_{ij}\neq 0}
\frac{\tilde{y}_i x_{ij}}
{\left\|\mathbf{x}_i-\mathbf{m}\right\|_2}
-
m_j
\sum_{i=1}^{N}
\frac{\tilde{y}_i}
{\left\|\mathbf{x}_i-\mathbf{m}\right\|_2}
$$

where:

- $r_j$ is the correlation associated with feature $j$;
- $x_{ij}$ is the TF-IDF value of feature $j$ in review $i$;
- $\tilde{y}_i=y_i/\|\mathbf{y}\|_2$ is the L2-normalised rating;
- $\mathbf{m}$ is the mean TF-IDF vector; and
- $N$ is the number of training reviews.

The first term sums the contributions from reviews where feature $j$
has a non-zero TF-IDF value.

The second term adjusts for the centred zero-valued features. Its inner
sum is shared by every feature, so it is calculated once from the
MapReduce document-statistics output and passed to the final correlation
job.

In [65]:
bonus_document_stats = pd.read_csv(
    "outputs/bonus_document_stats.tsv",
    sep="\t",
    header=None,
    names=[
        "document_id",
        "rating",
        "centred_feature_norm",
        "normalised_rating",
        "baseline_component"
    ]
)


bonus_baseline_sum = (
    bonus_document_stats[
        "baseline_component"
    ].sum()
)


with open(
    "outputs/bonus_baseline_sum.txt",
    "w",
    encoding="utf-8"
) as baseline_file:

    baseline_file.write(
        str(bonus_baseline_sum)
    )


print(
    "Document statistics:",
    len(bonus_document_stats)
)

print(
    "Baseline sum:",
    bonus_baseline_sum
)

print(
    "Zero feature norms:",
    (
        bonus_document_stats[
            "centred_feature_norm"
        ] == 0
    ).sum()
)

Document statistics: 40
Baseline sum: 25.334479465099996
Zero feature norms: 0


#### B.8 MapReduce Job 4: Final Correlation Vector

The final MapReduce job calculates one correlation value for every word.

The mapper joins each training TF-IDF value with:

- the L2-normalised rating of its review; and
- the centred L2 norm of its review.

The reducer sums the non-zero feature contributions and adjusts them for
the centred zero values using the feature mean and the shared baseline
sum.

Test records are emitted as markers so that test-only words remain in the
complete $D$-dimensional correlation vector.

In [66]:
%%writefile bonus_correlation_mapper.py
#!/usr/bin/env python3

import sys


# Load MapReduce document statistics.
document_statistics = {}

with open(
    "bonus_document_stats.tsv",
    "r",
    encoding="utf-8"
) as document_file:

    for line in document_file:

        fields = line.strip().split("\t")

        if len(fields) != 5:
            continue

        document_id = fields[0]

        centred_feature_norm = float(
            fields[2]
        )

        normalised_rating = float(
            fields[3]
        )

        document_statistics[document_id] = (
            centred_feature_norm,
            normalised_rating
        )


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 7:
        continue

    document_id = fields[0]
    word = fields[1]

    try:
        tfidf = float(fields[6])

    except ValueError:
        continue

    if document_id.startswith("train/"):

        statistics = document_statistics.get(
            document_id
        )

        if statistics is None:
            continue

        centred_feature_norm = statistics[0]
        normalised_rating = statistics[1]

        if centred_feature_norm > 0:

            contribution = (
                normalised_rating
                * tfidf
                / centred_feature_norm
            )

        else:

            contribution = 0.0

        print(
            word
            + "\t"
            + "DATA"
            + "\t"
            + str(contribution)
        )

    else:

        # Preserve words appearing only in the test set.
        print(
            word
            + "\t"
            + "MARKER"
            + "\t"
            + "0"
        )

Overwriting bonus_correlation_mapper.py


In [67]:
%%writefile bonus_correlation_reducer.py
#!/usr/bin/env python3

import os
import sys


baseline_sum = float(
    os.environ.get(
        "BASELINE_SUM",
        "0"
    )
)


# Load the MapReduce feature means.
feature_means = {}

with open(
    "bonus_feature_means.tsv",
    "r",
    encoding="utf-8"
) as mean_file:

    for line in mean_file:

        fields = line.strip().split("\t")

        if len(fields) != 2:
            continue

        feature_means[fields[0]] = float(
            fields[1]
        )


def print_correlation(
    word,
    contribution_sum,
    training_document_frequency
):

    feature_mean = feature_means.get(
        word,
        0.0
    )

    correlation = (
        contribution_sum
        - (
            feature_mean
            * baseline_sum
        )
    )

    print(
        word
        + "\t"
        + str(training_document_frequency)
        + "\t"
        + format(feature_mean, ".10f")
        + "\t"
        + format(correlation, ".10f")
    )


current_word = None
current_contribution_sum = 0.0
current_training_frequency = 0


for line in sys.stdin:

    line = line.strip()

    if line == "":
        continue

    fields = line.split("\t")

    if len(fields) != 3:
        continue

    word = fields[0]
    record_type = fields[1]

    try:
        contribution = float(fields[2])

    except ValueError:
        continue

    if word == current_word:

        if record_type == "DATA":

            current_contribution_sum += contribution
            current_training_frequency += 1

    else:

        if current_word is not None:

            print_correlation(
                current_word,
                current_contribution_sum,
                current_training_frequency
            )

        current_word = word
        current_contribution_sum = 0.0
        current_training_frequency = 0

        if record_type == "DATA":

            current_contribution_sum = contribution
            current_training_frequency = 1


if current_word is not None:

    print_correlation(
        current_word,
        current_contribution_sum,
        current_training_frequency
    )

Overwriting bonus_correlation_reducer.py


In [68]:
%%bash

INPUT_PATH="/users/hadoop/COMP3002/job2_tfidf"

OUTPUT_PATH="/users/hadoop/COMP3002/bonus_feature_correlation"


BASELINE_SUM=$(cat \
    outputs/bonus_baseline_sum.txt)


STREAMING_JAR=$(find \
    /home/hadoop/share/hadoop/tools/lib \
    -name "hadoop-streaming-*.jar" \
    | head -n 1)


hdfs dfs -rm -r -f "${OUTPUT_PATH}"


hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.job.reduces=1 \
    -files bonus_correlation_mapper.py,bonus_correlation_reducer.py,outputs/bonus_document_stats.tsv#bonus_document_stats.tsv,outputs/bonus_feature_means.tsv#bonus_feature_means.tsv \
    -cmdenv BASELINE_SUM="${BASELINE_SUM}" \
    -mapper "python3 bonus_correlation_mapper.py" \
    -reducer "python3 bonus_correlation_reducer.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}"

Deleted /users/hadoop/COMP3002/bonus_feature_correlation
packageJobJar: [/tmp/hadoop-unjar2096857209180732569/] [] /tmp/streamjob8469324165987513689.jar tmpDir=null


2026-07-17 20:43:58,631 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:43:59,072 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-07-17 20:43:59,626 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784262443103_0016
2026-07-17 20:44:01,075 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-17 20:44:01,246 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-17 20:44:01,581 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784262443103_0016
2026-07-17 20:44:01,581 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-17 20:44:02,289 INFO conf.Configuration: resource-types.xml not found
2026-07-17 20:44:02,292 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-17 20:44:02,532 INFO impl.YarnClientImpl: Submitted application application_17842624431

#### B.9 Validating the Correlation Vector

The final Hadoop result is copied to the local project.

The validation confirms that:

- one correlation record exists for every vocabulary word;
- the result has the same dimension as the Task 1 word list;
- no missing correlation values are present; and
- the values can be aligned with the original TF-IDF column order.

In [69]:
%%bash

HDFS_OUTPUT="/users/hadoop/COMP3002/bonus_feature_correlation"

LOCAL_OUTPUT="outputs/bonus_feature_correlations.tsv"


rm -f "${LOCAL_OUTPUT}"

hdfs dfs -getmerge \
    "${HDFS_OUTPUT}" \
    "${LOCAL_OUTPUT}"


echo "Number of correlation records:"

wc -l < "${LOCAL_OUTPUT}"


echo
echo "First 10 records:"

awk 'NR <= 10 {print}' "${LOCAL_OUTPUT}"


echo
echo "Records with an incorrect number of fields:"

awk -F '\t' '
    NF != 4 {
        invalid = invalid + 1
    }

    END {
        print invalid + 0
    }
' "${LOCAL_OUTPUT}"

Number of correlation records:
3127

First 10 records:
a	37	0.0313682500	0.0414844308
abandoned	1	0.0003903000	0.0048814884
abducted	1	0.0002516000	0.0046803136
ability	1	0.0002754500	-0.0051781316
able	1	0.0005764500	0.0068953167
about	19	0.0052846250	0.0085059260
above	2	0.0012330250	-0.0048477441
absence	1	0.0007218500	-0.0149809273
absolutely	1	0.0007314750	0.0113331821
absorb	0	0.0000000000	0.0000000000

Records with an incorrect number of fields:
0


In [70]:
bonus_feature_correlations = pd.read_csv(
    "outputs/bonus_feature_correlations.tsv",
    sep="\t",
    header=None,
    names=[
        "word",
        "training_document_frequency",
        "feature_mean",
        "correlation"
    ]
)


# Align the MapReduce result with the Task 1 word list.
bonus_correlation_series = (
    bonus_feature_correlations
    .set_index("word")["correlation"]
    .reindex(wordlist)
)


missing_correlations = (
    bonus_correlation_series
    .isna()
    .sum()
)


bonus_correlation_series = (
    bonus_correlation_series
    .fillna(0.0)
)


bonus_correlation_vector = (
    bonus_correlation_series
    .to_numpy()
)


print(
    "Correlation records:",
    len(bonus_feature_correlations)
)

print(
    "Vocabulary size:",
    len(wordlist)
)

print(
    "Correlation vector shape:",
    bonus_correlation_vector.shape
)

print(
    "Missing correlations:",
    missing_correlations
)

Correlation records: 3127
Vocabulary size: 3127
Correlation vector shape: (3127,)
Missing correlations: 0


#### B.10 Strongest Feature–Rating Associations

A positive correlation indicates that a word's TF-IDF feature tends to
be associated with higher review ratings.

A negative correlation indicates that the feature tends to be associated
with lower review ratings.

The following tables display the ten strongest positive and negative
correlations produced by Hadoop.

In [71]:
strongest_positive_correlations = (
    bonus_feature_correlations
    .sort_values(
        by="correlation",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)


strongest_negative_correlations = (
    bonus_feature_correlations
    .sort_values(
        by="correlation",
        ascending=True
    )
    .head(10)
    .reset_index(drop=True)
)


print("Strongest positive correlations:")

display(
    strongest_positive_correlations
)


print("Strongest negative correlations:")

display(
    strongest_negative_correlations
)

Strongest positive correlations:


,word,training_document_frequency,feature_mean,correlation
0,williams,12,0.008988,0.133101
1,collette,8,0.004486,0.076598
2,robin,11,0.005338,0.065166
3,gabriel,6,0.002997,0.060574
4,has,13,0.005278,0.059183
5,listener,6,0.002965,0.054767
6,night,8,0.003200,0.054056
7,story,17,0.005656,0.050047
8,his,21,0.009092,0.049637
9,toni,7,0.002717,0.049348


Strongest negative correlations:


,word,training_document_frequency,feature_mean,correlation
0,out,21,0.009547,-0.085565
1,comedy,5,0.004948,-0.077936
2,allen,3,0.005080,-0.077911
3,charlie,2,0.003581,-0.070587
4,woody,7,0.004159,-0.068267
5,this,31,0.016195,-0.065103
6,bergman,4,0.003851,-0.059645
7,teens,3,0.004066,-0.055989
8,only,15,0.004907,-0.055340
9,ghost,3,0.003875,-0.054504


In [72]:
# Save the complete MapReduce correlation result.
bonus_feature_correlations.to_csv(
    "outputs/bonus_feature_correlations.csv",
    index=False
)


# Save the aligned D-dimensional vector.
np.save(
    "outputs/bonus_correlation_vector.npy",
    bonus_correlation_vector
)


# Save the strongest results.
strongest_positive_correlations.to_csv(
    "outputs/bonus_strongest_positive.csv",
    index=False
)

strongest_negative_correlations.to_csv(
    "outputs/bonus_strongest_negative.csv",
    index=False
)


print("Bonus results saved successfully.")

Bonus results saved successfully.


#### Interpretation

The four-job Hadoop Streaming workflow produced 3,127 feature–rating
correlation records, matching the 3,127 words in the Task 1 vocabulary.
The aligned correlation vector therefore has shape `(3127,)`, with no
missing or malformed values.

The strongest positive association was found for `williams`, with a
correlation of approximately 0.1331. Other relatively strong positive
features included `collette`, `robin`, `gabriel` and `has`.

The strongest negative association was found for `out`, with a
correlation of approximately -0.0856. Other relatively strong negative
features included `comedy`, `allen`, `charlie` and `woody`.

Several of the strongest features are names rather than general
sentiment words. This suggests that the results are influenced by the
specific films, actors or topics appearing in the small sample. The
correlations should therefore not be interpreted as universal word
sentiment scores or as evidence of causation.

The TF-IDF vectors were centred and normalised review by review. The
rating vector was normalised using its uncentred L2 norm, following the
formula specified in the assignment.

The calculation used the sparse Hadoop TF-IDF output and four MapReduce
jobs. The dense `training_data` matrix was not used to compute the
correlations.

### Bonus Task Conclusion

The optional bonus task was completed successfully using Hadoop
Streaming.

The workflow calculated:

1. rating summary statistics and the uncentred rating-vector L2 norm;
2. the mean TF-IDF vector across the training reviews;
3. centred L2 norms for all 40 training-review feature vectors; and
4. the final feature–rating correlation vector.

The result contains 3,127 correlation values, exactly matching the Task 1
vocabulary. No missing or malformed correlation records were found.

This completes the required MapReduce implementation of the bonus
feature–response correlation analysis.

## Overall Marking Criteria for All Tasks

Your program will be marked against both functional and operational requirements. Functional requirements accounts for 80% of the mark, which measure how well your program achieves the expected functionalities and are further broken down into the items listed in marking schemes in those tasks. 

In addition to function requirements, your program should also meet the operational and style requirements, which can be broken down into the following. 

- Readability (5%): Comments should be included in your program to explain the main idea of your design; use meaningful variable and function names; do not declare variables that are not used in the program 
- Modularity (10%): Your program should make use of functions or classes wherever possible to achieve modular design and maximise reusability. 
- Useability (5%): Your program should be easy to use by the user. These include displaying messages for user interaction, performing adequate input validation, and allowing the users to choose the locations of the data file for model training. 

## Submission
Your python code solution including Hadoop streaming commands if any, documentation such as how to use the functions must be written in _this jupyter notebook_ with clear indication which is for which. Rename it to `COMP3002_assignment_yourstudentid.ipynb` and work on it. Add code and markdown blocks as you like. Submit your complete notebook through vUWS before deadline.